# Fashion Hypergraph Neural Network - Local Training
## Organized & Efficient Training Pipeline

This notebook implements a complete training pipeline for a Hypergraph Neural Network applied to fashion outfit compatibility prediction. All code is optimized to run on local machines.

## 1. Setup and Configuration

In [2]:
import os
import sys
import json
import random
import ast
import pickle
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import scipy.sparse as sp

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, auc, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, log_loss, mean_squared_error,
    precision_recall_curve, precision_score, recall_score, roc_auc_score
)

# ============================================
# 1. CONFIGURATION
# ============================================

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Paths - CONFIGURE THESE FOR YOUR LOCAL MACHINE
BASE_PATH = Path("c:/TFM/APP")  # Change this to your workspace root
DATA_FOLDER = BASE_PATH / "ml_pipeline/data"  # Where your CSV files are
CHECKPOINTS_PATH = BASE_PATH / "ml_pipeline/experiments_fashion_score"  # Where models will be saved
CHECKPOINTS_PATH.mkdir(parents=True, exist_ok=True)

# Dataset paths
TRAIN_OUTFITS_PATH = DATA_FOLDER / "train_outfits.csv"
TEST_OUTFITS_PATH = DATA_FOLDER / "test_outfits.csv"
TRAIN_ITEMS_PATH = DATA_FOLDER / "train_items.csv"
TEST_ITEMS_PATH = DATA_FOLDER / "test_items.csv"


META_PATH = CHECKPOINTS_PATH / "mapping_meta.json"
ENCODER_PATH = CHECKPOINTS_PATH / "attribute_encoder.pth"

print(f"Data folder: {DATA_FOLDER}")
print(f"Checkpoints folder: {CHECKPOINTS_PATH}")

# Random seed for reproducibility
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print("Random seed set to 42")

Using device: cuda
Data folder: c:\TFM\APP\ml_pipeline\data
Checkpoints folder: c:\TFM\APP\ml_pipeline\experiments_fashion_score
Random seed set to 42


## 2. Data Loading and Preprocessing

In [3]:
def ensure_list_and_array_columns(df, list_cols=None, array_cols=None):
    """Convert stringified list/array columns into Python lists/numpy arrays."""
    if list_cols is None:
        list_cols = []
    if array_cols is None:
        array_cols = []

    for col in list_cols:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: ast.literal_eval(x) if isinstance(x, str) else x
            )

    for col in array_cols:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: np.fromstring(x.strip("[]"), sep=" ") if isinstance(x, str) else x
            )

    return df

def load_or_build_mapping(df_items, mapping_path):
    """Load or build item_id to node_id mapping for a wardrobe."""
    if mapping_path.exists():
        try:
            with open(mapping_path, "r") as f:
                img2id = json.load(f)
            print(f"Loaded existing {mapping_path} mapping with {len(img2id)} items")
            return img2id
        except (json.JSONDecodeError, IOError) as e:
            print(f"⚠️  Warning: Could not load {mapping_path} ({e}). Rebuilding...")
            # Delete corrupted file and rebuild
            mapping_path.unlink(missing_ok=True)

    img_paths = df_items['item_id'].astype(str).unique().tolist()
    img2id = {p: i for i, p in enumerate(img_paths)}
    
    mapping_path.parent.mkdir(parents=True, exist_ok=True)
    with open(mapping_path, "w") as f:
        json.dump(img2id, f, indent=2)

    
    print(f"Created {mapping_path} mapping with {len(img2id)} items")
    return img2id

def add_node_ids(df_clothes, itemid_to_nodeid):
    """Add node_id column based on item_id mapping."""
    df_clothes = df_clothes.copy()
    df_clothes["item_id"] = df_clothes["item_id"].astype(str)
    itemid_to_nodeid = {str(k): v for k, v in itemid_to_nodeid.items()}
    df_clothes["node_id"] = df_clothes["item_id"].map(itemid_to_nodeid)
    df_clothes = df_clothes.drop(columns=["item_id"])
    return df_clothes

def convert_outfit_itemids_to_nodeids(df_outfits, itemid_to_nodeid):
    """Convert item_ids lists to node_ids lists in outfits dataframe."""
    df_outfits = df_outfits.copy()
    itemid_to_nodeid = {str(k): v for k, v in itemid_to_nodeid.items()}
    
    def convert_list(item_ids_list):
        """Convert list of item_ids to list of node_ids."""
        if not isinstance(item_ids_list, list):
            return []
        node_ids = []
        for item_id in item_ids_list:
            node_id = itemid_to_nodeid.get(str(item_id))
            if node_id is not None:
                node_ids.append(node_id)
        return node_ids
    
    df_outfits["item_ids"] = df_outfits["item_ids"].apply(convert_list)
    df_outfits = df_outfits.rename(columns={"item_ids": "node_ids"})
    return df_outfits

def load_and_process_data(outfits_path, items_path, wardrobe_name=""):
    """
    Load outfit and item CSVs, build mappings, convert to node_ids.
    
    Args:
        outfits_path: Path to outfits CSV
        items_path: Path to items CSV
        wardrobe_name: Name of wardrobe 
    
    Returns:
        tuple: (df_outfits, df_items, img2id_mapping)
    """
    outfits_path = Path(outfits_path)
    

    print(f"\n{'='*60}")
    print(f"Loading {wardrobe_name.upper()} data...")
    print(f"{'='*60}")
    
    # Load CSVs
    df_outfits = pd.read_csv(outfits_path)
    df_items = pd.read_csv(items_path)
    print(f"Loaded: {len(df_outfits)} outfits, {len(df_items)} items")
    
    # Fix list/array columns
    list_cols_items = ["related_indices", "category_indices", "main_category_indices", "sub_category_indices"]
    array_cols_items = ["img_embedding"]
    df_items = ensure_list_and_array_columns(df_items, list_cols_items, array_cols_items)
    
    list_cols_outfits = ["item_ids"]
    df_outfits = ensure_list_and_array_columns(df_outfits, list_cols_outfits)
    
    # Build mapping
    mapping_path = DATA_FOLDER / f"img2id_{wardrobe_name}.json"
    img2id = load_or_build_mapping(df_items, mapping_path)
    
    # Add node_ids to items
    df_items = add_node_ids(df_items, img2id)
    print(f"Added node_ids to items: {len(img2id)} unique nodes (0 to {len(img2id)-1})")
    
    # Convert outfit item_ids to node_ids
    df_outfits = convert_outfit_itemids_to_nodeids(df_outfits, img2id)
    print(f"Converted {len(df_outfits)} outfits to use node_ids")
    print(f"Sample outfit node_ids: {df_outfits['node_ids'].iloc[0]}")
    
    return df_outfits, df_items, img2id

In [4]:
# Load and process train and test data
df_train_outfits, df_train_items, img2id_train = load_and_process_data(
    TRAIN_OUTFITS_PATH, TRAIN_ITEMS_PATH, "train"
)


Loading TRAIN data...
Loaded: 33990 outfits, 123787 items
Loaded existing c:\TFM\APP\ml_pipeline\data\img2id_train.json mapping with 123787 items
Added node_ids to items: 123787 unique nodes (0 to 123786)
Converted 33990 outfits to use node_ids
Sample outfit node_ids: [580, 100496, 117071, 104208, 6335, 73980]


## 3. Feature Engineering

In [5]:
class AttributeEncoder(nn.Module):
    """Encodes categorical attributes into normalized embeddings."""
    def __init__(self, input_dim, output_dim=256, hidden_dim=512):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        x = self.encoder(x)
        return F.normalize(x, p=2, dim=-1)

def generate_attribute_embeddings(df, encoder=None, device='cpu'):
    """Convert categorical indices to attribute embeddings."""
    def parse_list_value(v):
        """Parse a value that might be string, list, or empty."""
        if isinstance(v, (list, np.ndarray)):
            return list(v)
        if isinstance(v, str):
            if v.strip() == "" or v.strip() == "[]":
                return []
            try:
                return ast.literal_eval(v)
            except Exception:
                return []
        return []
    
    def encode_list_column(col, max_index):
        binary_matrix = np.zeros((len(df), max_index), dtype=np.float32)
        for i, values in enumerate(df[col]):
            parsed_values = parse_list_value(values)
            for v in parsed_values:
                try:
                    v = int(v)
                    if 0 <= v < max_index:
                        binary_matrix[i, v] = 1.0
                except (ValueError, TypeError):
                    pass
        return binary_matrix

    max_related = max([max(parse_list_value(v)) if len(parse_list_value(v)) > 0 else 0 for v in df["related_indices"]]) + 1
    max_category = max([max(parse_list_value(v)) if len(parse_list_value(v)) > 0 else 0 for v in df["category_indices"]]) + 1
    max_main = max([max(parse_list_value(v)) if len(parse_list_value(v)) > 0 else 0 for v in df["main_category_indices"]]) + 1
    max_sub = max([max(parse_list_value(v)) if len(parse_list_value(v)) > 0 else 0 for v in df["sub_category_indices"]]) + 1

    related_bin = encode_list_column("related_indices", max_related)
    category_bin = encode_list_column("category_indices", max_category)
    main_bin = encode_list_column("main_category_indices", max_main)
    sub_bin = encode_list_column("sub_category_indices", max_sub)

    X_attr = np.concatenate([related_bin, category_bin, main_bin, sub_bin], axis=1)
    input_dim = X_attr.shape[1]

    encoder = AttributeEncoder(input_dim=input_dim).to(device)


    X_tensor = torch.tensor(X_attr, dtype=torch.float32).to(device)
    encoder.eval()
    with torch.no_grad():
        embeddings = encoder(X_tensor).cpu().numpy()

    return embeddings, encoder


In [6]:
print("Generating attribute embeddings for training items...")
attribute_embeddings_train, attribute_encoder = generate_attribute_embeddings(
    df_train_items, device=device
)
df_train_items["Xa"] = list(attribute_embeddings_train)

print(f"✓ Attribute embeddings generated successfully")
print(f"  Shape: {attribute_embeddings_train.shape}")
print(f"  Embedding dimension: {attribute_embeddings_train.shape[1]}")
print(f"  Number of items: {attribute_embeddings_train.shape[0]}")
print(f"  Sample embedding (first 10 values): {attribute_embeddings_train[0][:10]}")


Generating attribute embeddings for training items...
✓ Attribute embeddings generated successfully
  Shape: (123787, 256)
  Embedding dimension: 256
  Number of items: 123787
  Sample embedding (first 10 values): [-0.04048742 -0.01729244 -0.07893835  0.01338589 -0.0711981   0.04334889
  0.01718386 -0.09986281 -0.00947371  0.09259556]


## 4. Graph Construction

In [7]:
def build_hyperedges_from_outfits(df_outfits, wardrobe_name="", outfits_col="node_ids"):
    """Build hyperedges (outfits as hyperedges) from outfit DataFrame.
    
    Expects item_ids column to contain node_ids (already converted).
    Automatically loads from file if exists, otherwise builds and saves.
    
    Args:
        df_outfits: DataFrame with outfit node_ids
        wardrobe_name: Name for this wardrobe (used for save/load). Default: "" (empty string)
        outfits_col: Column name containing node_ids (default: "node_ids")
    
    Returns:
        edge_list: List of hyperedges (each hyperedge is a list of node_ids)
    """
    # Load from file if exists
    hyperedges_path = DATA_FOLDER / f"hyperedges_{wardrobe_name}.pkl"
    if hyperedges_path.exists():
        with open(hyperedges_path, "rb") as f:
            edge_list = pickle.load(f)
        print(f"Loaded existing hyperedges for '{wardrobe_name}'")
        return edge_list
    
    # Build hyperedges
    edge_list = []
    for node_ids in df_outfits[outfits_col]:
        if isinstance(node_ids, str):
            try:
                node_ids = ast.literal_eval(node_ids)
            except Exception:
                continue
        
        if isinstance(node_ids, list) and len(node_ids) >= 2:
            edge_list.append(node_ids)
    
    # Save to file
    with open(hyperedges_path, "wb") as f:
        pickle.dump(edge_list, f)
    print(f"Built and saved hyperedges for '{wardrobe_name}'")
    
    return edge_list

def build_incidence_matrix(num_nodes, hyperedges, device='cpu'):
    """Build hypergraph incidence matrix H (sparse) and compute degree matrices efficiently."""
    M = len(hyperedges)
    N = num_nodes
    
    if M == 0:
        # Return empty sparse matrices
        H = torch.sparse_coo_tensor(torch.empty((2, 0), dtype=torch.long), 
                                     torch.empty(0), (N, 0), device=device)
        # Return degree vectors as 1D tensors (not full matrices)
        Dv_inv_sqrt = torch.ones(N, device=device)
        De_inv = torch.empty(0, device=device)
        return H, Dv_inv_sqrt, De_inv
    
    # Build sparse incidence matrix using COO format - vectorized
    rows, cols = [], []
    for j, hedge in enumerate(hyperedges):
        for node in hedge:
            if 0 <= node < N:
                rows.append(node)
                cols.append(j)
    
    if len(rows) == 0:
        H = torch.sparse_coo_tensor(torch.empty((2, 0), dtype=torch.long),
                                     torch.empty(0), (N, M), device=device)
        Dv_inv_sqrt = torch.ones(N, device=device)
        De_inv = torch.ones(M, device=device)
        return H, Dv_inv_sqrt, De_inv
    
    # Create sparse tensor
    rows_t = torch.tensor(rows, dtype=torch.long, device=device)
    cols_t = torch.tensor(cols, dtype=torch.long, device=device)
    indices = torch.stack([rows_t, cols_t])
    values = torch.ones(len(rows), device=device)
    H = torch.sparse_coo_tensor(indices, values, (N, M), device=device)
    
    # Compute degree vectors using bincount (vectorized)
    dv = torch.bincount(rows_t, minlength=N).float().to(device)
    de = torch.bincount(cols_t, minlength=M).float().to(device)
    
    # Replace zeros with ones
    dv = torch.where(dv == 0, torch.ones_like(dv), dv)
    de = torch.where(de == 0, torch.ones_like(de), de)
    
    # Return as inverse sqrt and inverse (as vectors, not full matrices)
    Dv_inv_sqrt = torch.pow(dv, -0.5)
    De_inv = torch.pow(de, -1.0)
    
    return H, Dv_inv_sqrt, De_inv


In [8]:
# Build hypergraph from outfits
print("Building hypergraph...")
edge_list_train = build_hyperedges_from_outfits(df_train_outfits, wardrobe_name="train", outfits_col="node_ids")
print(f"Found {len(edge_list_train)} hyperedges (outfits)")

Building hypergraph...
Loaded existing hyperedges for 'train'
Found 33990 hyperedges (outfits)


In [9]:
# Build incidence matrix
N_nodes_train = len(img2id_train)
H_train, Dv_inv_sqrt_train, De_inv_train = build_incidence_matrix(N_nodes_train, edge_list_train, device=device)
print(f"Incidence matrix H_train shape: {H_train.shape}")
print(f"Dv_inv_sqrt_train shape: {Dv_inv_sqrt_train.shape}")
print(f"De_inv_train shape: {De_inv_train.shape}")
print(f"H_train is sparse: {H_train.is_sparse}")

Incidence matrix H_train shape: torch.Size([123787, 33990])
Dv_inv_sqrt_train shape: torch.Size([123787])
De_inv_train shape: torch.Size([33990])
H_train is sparse: True


## 5. Model Architecture

In [10]:
class MultiHeadAttnPool(nn.Module):
    """Multi-head attention pooling over items in an outfit."""
    def __init__(self, d_model, n_heads=4, d_k=None, d_v=None, dropout=0.1, out_dim=None):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_k or (d_model // n_heads)
        self.d_v = d_v or (d_model // n_heads)
        
        self.q_proj = nn.Linear(d_model, n_heads * self.d_k)
        self.k_proj = nn.Linear(d_model, n_heads * self.d_k)
        self.v_proj = nn.Linear(d_model, n_heads * self.d_v)
        self.out_dim = out_dim or d_model
        self.out_proj = nn.Linear(n_heads * self.d_v, self.out_dim)
        self.dropout = nn.Dropout(dropout)
        self.global_query = nn.Parameter(torch.randn(1, d_model) * 0.02)

    def forward(self, emb, mask):
        B, L, D = emb.shape
        q = self.global_query.expand(B, -1)
        q = self.q_proj(q)
        k = self.k_proj(emb)
        v = self.v_proj(emb)

        q = q.view(B, self.n_heads, self.d_k).unsqueeze(2)
        k = k.view(B, L, self.n_heads, self.d_k).permute(0, 2, 1, 3)
        v = v.view(B, L, self.n_heads, self.d_v).permute(0, 2, 1, 3)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None:
            mask_ = mask.unsqueeze(1).unsqueeze(1)
            scores = scores.masked_fill(mask_ == 0, float('-1e9'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        
        out = torch.matmul(attn, v)
        out = out.view(B, -1)
        out = self.out_proj(out)
        return out, attn.squeeze(2)


class HypergraphConvLayer(nn.Module):
    """Residual hypergraph convolution layer with layer normalization."""
    def __init__(self, in_dim, out_dim, use_bias=True):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=use_bias)
        self.norm = nn.LayerNorm(out_dim)
        self.res_proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else None

    def forward(self, X, H, Dv_inv_sqrt, De_inv):
        # Hypergraph convolution: D = H @ (De_inv @ (H.t() @ (Dv_inv_sqrt @ X)))
        # Step 1: Diagonal scaling Dv_inv_sqrt @ X
        A = Dv_inv_sqrt.unsqueeze(-1) * X  # (N, D)
        
        # Step 2: Sparse matrix H.t() @ A
        # Use torch.sparse.mm or alternative approach for sparse-dense multiplication
        if H.is_sparse:
            H_t = H.t()  # Transpose sparse tensor
            # Use spmm (sparse-dense) instead of mm for better stability
            B = torch.sparse.mm(H_t, A)  # (M, D)
        else:
            B = H.t() @ A
        
        # Step 3: Diagonal scaling De_inv @ B  
        C = De_inv.unsqueeze(-1) * B  # (M, D)
        
        # Step 4: Sparse matrix H @ C
        if H.is_sparse:
            D = torch.sparse.mm(H, C)  # (N, D)
        else:
            D = H @ C
        
        # Step 5: Diagonal scaling Dv_inv_sqrt @ D
        D = Dv_inv_sqrt.unsqueeze(-1) * D  # (N, D)
        
        out = self.linear(D)
        out = F.relu(out)
        res = self.res_proj(X) if self.res_proj is not None else X
        out = out + res
        out = self.norm(out)
        return out


class HGNN(nn.Module):
    """Hypergraph Neural Network with configurable depth and width."""
    def __init__(self, in_dim, hidden_dims=(256, 256), out_dim=128, dropout=0.2):
        super().__init__()
        dims = [in_dim] + list(hidden_dims)
        self.layers = nn.ModuleList([
            HypergraphConvLayer(dims[i], dims[i + 1]) 
            for i in range(len(dims) - 1)
        ])
        self.project = nn.Linear(dims[-1], out_dim)
        self.dropout = nn.Dropout(dropout)
        self.out_norm = nn.LayerNorm(out_dim)

    def forward(self, X, H, Dv_inv_sqrt, De_inv):
        x = X
        for layer in self.layers:
            x = layer(x, H, Dv_inv_sqrt, De_inv)
            x = self.dropout(x)
        x = self.project(x)
        x = F.relu(x)
        x = self.out_norm(x)
        x = F.normalize(x, p=2, dim=-1)
        return x


class OutfitScoreRegressor(nn.Module):
    """Regressor head for outfit score prediction."""
    def __init__(self, input_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.mlp(x).squeeze(-1)


class FashionHyperGraphModel(nn.Module):
    """
    Modelo V2 parametrizable para todas las variantes de Fashion HyperGraph.
    
    Parámetros:
        clip_embed_dim (int): dimensión de embeddings de imagen (CLIP).
        attr_embed_dim (int): dimensión de embeddings de atributos de prendas.
        fusion_hidden (int): tamaño del MLP de fusión entre imagen y atributos.
        final_embedding_dim (int): dimensión final de embeddings de nodos.
        dropout (float): tasa de dropout general en capas MLP y HGNN.
        attn_heads (int): número de cabezas de atención en MultiHeadAttnPool.
        use_hgnn (bool): si se utiliza HGNN o solo proyección simple.
        hgnn_hidden_list (list of tuple): lista de ramas HGNN, cada tupla define los hidden dims de esa rama.
        use_cross_attention (bool): aplica cross-attention entre nodos de outfit antes del pooling.
        use_hierarchical_pooling (bool): aplica pooling jerárquico por categorías de prendas.
        use_moe (bool): activa Mixture of Experts (MoE) sobre los nodos.
        num_experts (int): número de expertos si MoE está activo.
    """
    def __init__(self,
                 clip_embed_dim=512,
                 attr_embed_dim=256,
                 fusion_hidden=512,
                 final_embedding_dim=256,
                 dropout=0.2,
                 attn_heads=4,
                 use_hgnn=True,
                 hgnn_hidden_list=[(256, 256, 256)],  # lista de ramas HGNN
                 use_cross_attention=False,
                 use_hierarchical_pooling=False,
                 use_moe=False,
                 num_experts=1):
        super().__init__()
        self.use_hgnn = use_hgnn
        self.use_cross_attention = use_cross_attention
        self.use_hierarchical_pooling = use_hierarchical_pooling
        self.use_moe = use_moe
        self.num_experts = num_experts

        # --------------------------
        # Feature projections
        # --------------------------
        self.clip_proj = nn.Sequential(
            nn.Linear(clip_embed_dim, fusion_hidden//2),
            nn.ReLU()
        )
        self.attr_proj = nn.Sequential(
            nn.Linear(attr_embed_dim, fusion_hidden//2),
            nn.ReLU()
        )
        self.fusion_mlp = nn.Sequential(
            nn.Linear(fusion_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # --------------------------
        # Branch-specific projections
        # --------------------------
        self.branch_projs = nn.ModuleList()
        for hidden_dims in hgnn_hidden_list:
            proj = nn.Linear(fusion_hidden, hidden_dims[0])
            self.branch_projs.append(proj)

        # --------------------------
        # HGNN ramas
        # --------------------------
        self.hgnn_branches = nn.ModuleList()
        for hidden_dims in hgnn_hidden_list:
            if self.use_hgnn:
                branch = HGNN(in_dim=hidden_dims[0],
                              hidden_dims=hidden_dims,
                              out_dim=hidden_dims[-1],
                              dropout=dropout)
            else:
                branch = nn.Sequential(
                    nn.Linear(hidden_dims[0], hidden_dims[-1]),
                    nn.ReLU(),
                    nn.LayerNorm(hidden_dims[-1])
                )
            self.hgnn_branches.append(branch)

        # --------------------------
        # Mixture of Experts (MoE)
        # --------------------------
        if self.use_moe and self.num_experts > 1:
            # MoE experts take the concatenated branch outputs
            self.moe_gate_dim = sum([hd[-1] for hd in hgnn_hidden_list])
            self.moe_proj = nn.Linear(self.moe_gate_dim, hgnn_hidden_list[0][0])
            self.experts = nn.ModuleList([
                HGNN(hgnn_hidden_list[0][0], hgnn_hidden_list[0], final_embedding_dim, dropout)
                for _ in range(num_experts)
            ])
            self.router = nn.Sequential(
                nn.Linear(self.moe_gate_dim, num_experts),
                nn.Softmax(dim=-1)
            )

        # --------------------------
        # Cross-Attention
        # --------------------------
        if self.use_cross_attention:
            self.cross_attn = nn.MultiheadAttention(
                embed_dim=final_embedding_dim,
                num_heads=attn_heads,
                dropout=dropout,
                batch_first=True
            )

        # --------------------------
        # Proyector final
        # --------------------------
        total_dim = sum([hd[-1] for hd in hgnn_hidden_list])
        if self.use_moe and self.num_experts > 1:
            total_dim = final_embedding_dim
        self.project = nn.Linear(total_dim, final_embedding_dim)

        # --------------------------
        # Pooling
        # --------------------------
        if self.use_hierarchical_pooling:
            self.sub_pool_top = MultiHeadAttnPool(final_embedding_dim, n_heads=2, dropout=dropout)
            self.sub_pool_bottom = MultiHeadAttnPool(final_embedding_dim, n_heads=2, dropout=dropout)
            self.sub_pool_accessory = MultiHeadAttnPool(final_embedding_dim, n_heads=2, dropout=dropout)
        self.attn_pool = MultiHeadAttnPool(final_embedding_dim, n_heads=attn_heads, dropout=dropout)

        # --------------------------
        # Regressor
        # --------------------------
        self.regressor = OutfitScoreRegressor(
            input_dim=final_embedding_dim,
            hidden_dim=final_embedding_dim//2,
            dropout=dropout
        )

    def forward(self, clip_feats, attr_feats, H=None, Dv_inv_sqrt=None, De_inv=None,
                outfit_nodes=None, outfit_mask=None):
        # --------------------------
        # Feature projection y fusión
        # --------------------------
        c = self.clip_proj(clip_feats)
        a = self.attr_proj(attr_feats)
        fused = torch.cat([c, a], dim=1)
        x = self.fusion_mlp(fused)

        # --------------------------
        # Salida de ramas (con proyecciones específicas)
        # --------------------------
        branch_outputs = []
        for proj, branch in zip(self.branch_projs, self.hgnn_branches):
            x_proj = proj(x)  # Project to branch-specific input dimension
            if self.use_hgnn:
                out = branch(x_proj, H, Dv_inv_sqrt, De_inv)
            else:
                out = branch(x_proj)
            branch_outputs.append(out)
        node_emb = torch.cat(branch_outputs, dim=-1)

        # --------------------------
        # Mixture of Experts
        # --------------------------
        if self.use_moe and self.num_experts > 1:
            x_moe = self.moe_proj(node_emb)  # Project from branch outputs to expert input dimension
            expert_outputs = []
            for expert in self.experts:
                expert_out = expert(x_moe, H, Dv_inv_sqrt, De_inv)
                expert_outputs.append(expert_out)
            expert_stack = torch.stack(expert_outputs, dim=1)  # (N, num_experts, final_embedding_dim)
            routing_weights = self.router(node_emb)  # (N, num_experts)
            # Weighted sum: (N, num_experts, D) * (N, num_experts, 1) -> sum over experts
            weighted_experts = expert_stack * routing_weights.unsqueeze(-1)  # (N, num_experts, D)
            node_emb = weighted_experts.sum(dim=1)  # (N, D)

        # --------------------------
        # Proyector final y normalización
        # --------------------------
        node_emb = self.project(node_emb)
        node_emb = F.normalize(node_emb, p=2, dim=-1)

        # --------------------------
        # Cross-Attention
        # --------------------------
        if self.use_cross_attention and outfit_nodes is not None:
            emb = node_emb[outfit_nodes]
            emb, _ = self.cross_attn(
                emb, emb, emb,
                key_padding_mask=(1-outfit_mask).bool() if outfit_mask is not None else None
            )
            node_emb[outfit_nodes] = emb

        # --------------------------
        # Pooling jerárquico o estándar
        # --------------------------
        if outfit_nodes is None:
            return None, node_emb

        emb = node_emb[outfit_nodes]
        if self.use_hierarchical_pooling:
            top = self.sub_pool_top(emb, outfit_mask)[0]
            bottom = self.sub_pool_bottom(emb, outfit_mask)[0]
            accessory = self.sub_pool_accessory(emb, outfit_mask)[0]
            pooled = top + bottom + accessory
            attn = None
        else:
            pooled, attn = self.attn_pool(emb, outfit_mask)

        # --------------------------
        # Regressor final
        # --------------------------
        outfit_scores = self.regressor(pooled)
        return outfit_scores, node_emb, pooled, attn

print("Model architecture defined - Regressor is inside the model!")


Model architecture defined - Regressor is inside the model!


## 6. Training Infrastructure

In [ ]:
class NodeIdPaddedOutfitDataset(Dataset):
    """Dataset for outfits with padded node IDs."""
    def __init__(self, outfits_node_ids, labels, Xc, Xa, clothes_df=None, pad_node_id=0, max_len=None, device='cpu'):
        self.outfits = outfits_node_ids
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.pad_node_id = int(pad_node_id)
        self.max_len = max_len if max_len is not None else max(len(o) for o in outfits_node_ids)
        self.device = device
        self.Xc = torch.tensor(Xc, dtype=torch.float32, device=device)
        self.Xa = torch.tensor(Xa, dtype=torch.float32, device=device)
        self.clothes_df = clothes_df  # Store for category-based sorting

    def __len__(self):
        return len(self.outfits)

    def __getitem__(self, idx):
        node_ids = list(self.outfits[idx])
        
        # Sort items by category order: all-body(0), tops(2), bottoms(1), shoes(5), accessories
        if self.clothes_df is not None:
            category_order = {0: 0, 2: 1, 1: 2, 5: 3, 3: 4, 4: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10}
            
            def get_node_sort_key(node_id):
                # Find the row in clothes_df with this node_id
                rows = self.clothes_df[self.clothes_df["node_id"] == node_id]
                if len(rows) > 0:
                    main_cats = rows.iloc[0].get("main_category_indices", [999])
                    if isinstance(main_cats, list) and len(main_cats) > 0:
                        main_cat = main_cats[0]
                    else:
                        main_cat = 999
                else:
                    main_cat = 999
                return category_order.get(main_cat, 999)
            
            node_ids = sorted(node_ids, key=get_node_sort_key)
        
        label = self.labels[idx]
        pad_len = self.max_len - len(node_ids)
        nodes = node_ids + [self.pad_node_id] * pad_len
        mask = [1] * len(node_ids) + [0] * pad_len
        return {
            "nodes": torch.tensor(nodes, dtype=torch.long),
            "mask": torch.tensor(mask, dtype=torch.float32),
            "label": label,
            "orig_node_ids": node_ids
        }

def padded_collate(batch):
    """Collate function for DataLoader."""
    nodes = torch.stack([b["nodes"] for b in batch])
    masks = torch.stack([b["mask"] for b in batch])
    labels = torch.stack([b["label"] for b in batch])
    orig = [b["orig_node_ids"] for b in batch]
    return {"nodes": nodes, "mask": masks, "label": labels, "orig_node_ids": orig}

def _parse_embedding_cell(x):
    """Parse embedding from various formats."""
    if x is None:
        return None
    if isinstance(x, np.ndarray):
        return x.astype(np.float32)
    if isinstance(x, (list, tuple)):
        return np.array(x, dtype=np.float32)
    s = str(x).strip()
    if s == "":
        return None
    if s.startswith("[") and s.endswith("]"):
        try:
            return np.array(eval(s), dtype=np.float32)
        except Exception:
            return np.fromstring(s.strip("[]"), sep=",", dtype=np.float32)
    try:
        return np.array(eval(s), dtype=np.float32)
    except Exception:
        return np.fromstring(s, sep=",", dtype=np.float32)

def build_Xc_Xa_from_clothes_df(clothes_df, img_col="img_embedding", attr_col="attr_embedding", node_id_col="node_id"):
    """Build feature matrices from clothing dataframe."""
    node_ids = clothes_df[node_id_col].astype(int).values
    max_node = int(node_ids.max())
    N = max_node + 1

    first_row = clothes_df[clothes_df[img_col].notnull()].iloc[0]
    d_img = _parse_embedding_cell(first_row[img_col]).shape[0]
    d_attr = _parse_embedding_cell(first_row[attr_col]).shape[0]

    Xc = np.zeros((N, d_img), dtype=np.float32)
    Xa = np.zeros((N, d_attr), dtype=np.float32)

    for _, row in clothes_df.iterrows():
        nid = int(row[node_id_col])
        img = _parse_embedding_cell(row[img_col])
        attr = _parse_embedding_cell(row[attr_col])
        if img is not None:
            Xc[nid] = img
        if attr is not None:
            Xa[nid] = attr

    return Xc, Xa

def build_loaders_from_dataframes(
    clothes_df, sets_df,
    outfits_col="node_ids", label_col="label",
    img_col="img_embedding", attr_col="attr_embedding",
    node_id_col="node_id",
    val_split=0.1, pad_node_id=0,
    batch_size=64, device='cpu',
    random_state=42
):
    """Build train/val DataLoaders."""
    Xc, Xa = build_Xc_Xa_from_clothes_df(clothes_df, img_col=img_col, attr_col=attr_col, node_id_col=node_id_col)

    def _parse_node_list(x):
        if isinstance(x, (list, tuple, np.ndarray)):
            return [int(i) for i in x]
        try:
            return [int(i) for i in eval(x)]
        except Exception:
            return []
    
    outfits_node_ids = [_parse_node_list(x) for x in sets_df[outfits_col].values]
    labels = sets_df[label_col].values
    
    # Split data
    np.random.seed(random_state)
    n = len(outfits_node_ids)
    val_size = int(n * val_split)
    val_idxs = np.random.choice(n, val_size, replace=False)
    train_idxs = np.array([i for i in range(n) if i not in val_idxs])
    
    train_outfits = [outfits_node_ids[i] for i in train_idxs]
    train_labels = labels[train_idxs]
    val_outfits = [outfits_node_ids[i] for i in val_idxs]
    val_labels = labels[val_idxs]
    
    # Create datasets with clothes_df reference for category-based sorting
    train_ds = NodeIdPaddedOutfitDataset(
        train_outfits, train_labels, Xc, Xa, 
        clothes_df=clothes_df,
        pad_node_id=pad_node_id, 
        device=device
    )
    val_ds = NodeIdPaddedOutfitDataset(
        val_outfits, val_labels, Xc, Xa,
        clothes_df=clothes_df,
        pad_node_id=pad_node_id, 
        device=device
    )
    
    # Create loaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=padded_collate)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=padded_collate)
    
    return train_ds, val_ds, train_loader, val_loader

def masked_mean_pool(node_embeddings, nodes, mask):
    """Mean pooling with mask."""
    emb = node_embeddings[nodes]
    m = mask.unsqueeze(-1).to(emb.dtype)
    summed = (emb * m).sum(dim=1)
    counts = m.sum(dim=1).clamp(min=1.0)
    return summed / counts

def evaluate_with_embeddings(model, val_loader, device, H=None, Dv_inv_sqrt=None, De_inv=None, threshold=0.5):
    """Evaluate model on validation set. Regressor is inside the model."""
    model.eval()
    val_total = 0.0
    probs_list = []
    labels_list = []
    Xc = val_loader.dataset.Xc
    Xa = val_loader.dataset.Xa

    with torch.no_grad():
        for batch in val_loader:
            nodes = batch["nodes"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["label"].to(device)
            
            # Clamp node indices to valid range
            num_nodes = Xc.shape[0]
            nodes_clamped = torch.clamp(nodes, 0, num_nodes - 1)

            # Forward pass - model returns outfit scores directly
            outfit_scores, _, _, _ = model(Xc, Xa, H, Dv_inv_sqrt, De_inv,
                                          outfit_nodes=nodes_clamped, outfit_mask=mask)
            
            # Loss - scores already in [0, 1]
            loss = F.binary_cross_entropy(outfit_scores, labels).item()
            val_total += loss * len(labels)

            probs = outfit_scores.cpu().numpy()
            probs_list.append(probs)
            labels_list.append(labels.cpu().numpy())

    probs_all = np.concatenate(probs_list)
    labels_all = np.concatenate(labels_list)
    val_loss = val_total / len(val_loader.dataset)
    
    preds = (probs_all >= threshold).astype(int)

    metrics = {
        "val_loss": val_loss,
        "mse": mean_squared_error(labels_all, probs_all),
        "roc_auc": roc_auc_score(labels_all, probs_all) if len(np.unique(labels_all)) > 1 else float("nan"),
        "avg_score": float(probs_all.mean()),
        "accuracy": accuracy_score(labels_all, preds),
        "precision": precision_score(labels_all, preds, zero_division=0),
        "recall": recall_score(labels_all, preds, zero_division=0),
        "f1": f1_score(labels_all, preds, zero_division=0),
    }

    try:
        tn, fp, fn, tp = confusion_matrix(labels_all, preds).ravel()
        metrics["tp"], metrics["fp"], metrics["tn"], metrics["fn"] = int(tp), int(fp), int(tn), int(fn)
    except:
        metrics["tp"], metrics["fp"], metrics["tn"], metrics["fn"] = 0, 0, 0, 0

    model.train()
    return metrics, probs_all, labels_all

print("Training infrastructure defined")

Training infrastructure defined


## 7. Data Loading and Model Training

In [12]:
# Build train/val loaders
print("Building data loaders...")

train_ds, val_ds, train_loader, val_loader = build_loaders_from_dataframes(
    clothes_df=df_train_items,
    sets_df=df_train_outfits,
    outfits_col="node_ids",
    label_col="label",
    img_col="img_embedding",
    attr_col="Xa",
    node_id_col="node_id",
    val_split=0.1,
    pad_node_id=0,
    batch_size=252,
    device=device,
    random_state=42
)

print(f"Train dataset: {len(train_ds)}, Val dataset: {len(val_ds)}")
print(f"Feature shapes: Xc={train_ds.Xc.shape}, Xa={train_ds.Xa.shape}")

# Verify data loading
batch = next(iter(train_loader))
print(f"Batch shapes - nodes: {batch['nodes'].shape}, mask: {batch['mask'].shape}, label: {batch['label'].shape}")

Building data loaders...
Train dataset: 30591, Val dataset: 3399
Feature shapes: Xc=torch.Size([123787, 512]), Xa=torch.Size([123787, 256])
Batch shapes - nodes: torch.Size([252, 16]), mask: torch.Size([252, 16]), label: torch.Size([252])


In [ ]:
def train_outfit_regressor_hybrid(
    model, optimizer, train_loader, val_loader,
    H, Dv_inv_sqrt, De_inv,
    device="cpu", epochs=20,
    threshold=0.5,
    early_stop_patience=10, early_stop_min_delta=0.0
):
    """Main training loop with early stopping."""
    from tqdm import tqdm
    import time
    
    model.to(device)
    Xc = train_loader.dataset.Xc.to(device) if hasattr(train_loader.dataset.Xc, 'to') else torch.tensor(train_loader.dataset.Xc, dtype=torch.float32, device=device)
    Xa = train_loader.dataset.Xa.to(device) if hasattr(train_loader.dataset.Xa, 'to') else torch.tensor(train_loader.dataset.Xa, dtype=torch.float32, device=device)
    
    # Move hypergraph tensors to device
    H = H.to(device) if hasattr(H, 'to') else H
    Dv_inv_sqrt = Dv_inv_sqrt.to(device) if hasattr(Dv_inv_sqrt, 'to') else Dv_inv_sqrt
    De_inv = De_inv.to(device) if hasattr(De_inv, 'to') else De_inv
    
    # Get number of nodes from Xc shape
    num_nodes = Xc.shape[0]

    best_val_loss = float("inf")
    best_state = None
    best_metrics = {}
    num_bad_epochs = 0

    history = {
        "train_loss": [], "val_loss": [], "mse": [], "roc_auc": [],
        "accuracy": [], "f1": []
    }

    print(f"\n{'='*100}")
    print(f"Training on {device} | Epochs: {epochs} | Batches/epoch: {len(train_loader)}")
    print(f"{'='*100}\n")

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        
        # Progress bar for batches
        pbar = tqdm(train_loader, desc=f"Epoch {epoch:3d}/{epochs}", unit="batch", leave=False)
        for batch_idx, batch in enumerate(pbar):
            optimizer.zero_grad()
            nodes = batch["nodes"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["label"].to(device)

            # Clamp node indices to valid range [0, num_nodes-1]
            nodes_clamped = torch.clamp(nodes, 0, num_nodes - 1)

            # Forward pass
            outfit_scores, _, _, _ = model(Xc, Xa, H, Dv_inv_sqrt, De_inv,
                                           outfit_nodes=nodes_clamped, outfit_mask=mask)
            
            # Loss
            loss = F.binary_cross_entropy(outfit_scores, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
            
            # Update progress bar with loss
            pbar.set_postfix({"Loss": f"{loss.item():.6f}"})

        avg_train_loss = total_loss / len(train_loader.dataset)
        history["train_loss"].append(avg_train_loss)

        # Validation
        metrics, _, _ = evaluate_with_embeddings(model, val_loader, device, H=H, Dv_inv_sqrt=Dv_inv_sqrt, De_inv=De_inv, threshold=threshold)

        for k in ["val_loss", "mse", "roc_auc", "accuracy", "f1"]:
            if k in metrics:
                history[k].append(metrics[k])

        current_val_loss = float(metrics["val_loss"])
        
        # Track best val loss
        if current_val_loss < best_val_loss - early_stop_min_delta:
            best_val_loss = current_val_loss
            best_metrics = metrics.copy()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            num_bad_epochs = 0
        else:
            num_bad_epochs += 1

        # Print epoch summary (matching Fashion notebook format, adapted for outfit compatibility metrics)
        print(f"Epoch {epoch}/{epochs} | TrainLoss={avg_train_loss:.6f} | ValLoss={metrics['val_loss']:.6f} | BestVal={best_val_loss:.6f} | MSE={metrics['mse']:.6f} | ROC-AUC={metrics['roc_auc']:.4f} | F1={metrics['f1']:.4f}")
        
        if num_bad_epochs > early_stop_patience:
            print(f"\nEarly stopping at epoch {epoch}. Best val_loss: {best_val_loss:.6f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val_loss, best_metrics, history



In [ ]:
# Initialize model
print("\nInitializing model...")
model = FashionHyperGraphModel(
    clip_embed_dim=512,
    attr_embed_dim=256,
    fusion_hidden=512,
    hgnn_hidden_list=[(256, 256, 256),],
    final_embedding_dim=256,
    dropout=0.15,
    attn_heads=2,
    use_hierarchical_pooling=True
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-2)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Train the model
print("\nStarting training...")
model, best_val_loss, best_metrics, history = train_outfit_regressor_hybrid(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    H=H_train,
    Dv_inv_sqrt=Dv_inv_sqrt_train,
    De_inv=De_inv_train,
    device=device,
    epochs=100,  
    threshold=0.5,
    early_stop_patience=10,
    early_stop_min_delta=1e-4
)

print(f"\nTraining complete!")
print(f"Best validation loss: {best_val_loss:.6f}")
print(f"Best metrics: {best_metrics}")

In [ ]:
# Save the trained model
import os
from pathlib import Path

# Define checkpoint directory
checkpoint_dir = Path("../experiments_fashion_score")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Create experiment subdirectory with timestamp
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
exp_name = f"hgnn_f512_w256x256x256_h2_e256_dr15_adamw_lr5e-5_wd1e-2_{timestamp}"
exp_dir = checkpoint_dir / exp_name
exp_dir.mkdir(parents=True, exist_ok=True)

# Save model state dict
torch.save(model.state_dict(), exp_dir / "model.pt")
print(f"✓ Model checkpoint saved: {exp_dir / 'model.pt'}")

# Save training history
import json
with open(exp_dir / "history.json", "w") as f:
    history_json = {k: [float(v) for v in vals] for k, vals in history.items()}
    json.dump(history_json, f, indent=2)
print(f"✓ Training history saved: {exp_dir / 'history.json'}")

# Save best metrics
with open(exp_dir / "metrics.json", "w") as f:
    metrics_json = {k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                   for k, v in best_metrics.items()}
    json.dump(metrics_json, f, indent=2)
print(f"✓ Best metrics saved: {exp_dir / 'metrics.json'}")

# Save experiment summary
summary = {
    "name": exp_name,
    "timestamp": timestamp,
    "best_val_loss": float(best_val_loss),
    "best_metrics": {k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                    for k, v in best_metrics.items()},
    "model_config": {
        "clip_embed_dim": 512,
        "attr_embed_dim": 256,
        "fusion_hidden": 512,
        "hgnn_hidden_list": [(256, 256, 256)],
        "final_embedding_dim": 256,
        "dropout": 0.15,
        "attn_heads": 2,
        "use_hierarchical_pooling": True
    },
    "optimizer_config": {
        "type": "AdamW",
        "lr": 5e-5,
        "weight_decay": 1e-2,
        "betas": [0.9, 0.999]
    },
    "training_config": {
        "epochs": 100,
        "batch_size": 252,
        "threshold": 0.5,
        "early_stop_patience": 5,
        "early_stop_min_delta": 1e-4
    }
}

with open(exp_dir / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"✓ Summary saved: {exp_dir / 'summary.json'}")

print(f"\n{'='*70}")
print(f"Experiment saved to: {exp_dir}")
print(f"{'='*70}")

## 8. Save Model and Results

In [ ]:
def generate_experiment_name(model, optimizer, clip_embed_dim=512, attn_heads=4, 
                             final_embedding_dim=256, dropout=0.2, lr=1e-3, wd=0.0):
    """Generate standardized experiment name matching HGNN convention.
    
    Format: hgnn_f{clip_embed_dim}_w{hgnn_hidden}_h{attn_heads}_e{final_embedding_dim}_dr{dropout_int}_{optimizer}_lr{lr}_wd{wd}_b{beta1}_{beta2}
    Example: hgnn_f512_w256x256x256_h4_e256_dr20_adamw_lr1e-03_wd0e0_b09_0999
    """
    # Extract HGNN hidden dimensions from model
    hgnn_hidden = ""
    if hasattr(model, 'hgnn_hidden'):
        hgnn_hidden = "x".join(str(h) for h in model.hgnn_hidden)
    else:
        hgnn_hidden = "256x256x256"
    
    # Extract optimizer name and betas
    opt_name = "adam"
    beta1, beta2 = 0.9, 0.999
    if optimizer is not None:
        opt_class = optimizer.__class__.__name__.lower()
        if "adamw" in opt_class:
            opt_name = "adamw"
        elif "adam" in opt_class:
            opt_name = "adam"
        elif "sgd" in opt_class:
            opt_name = "sgd"
        
        # Extract betas from optimizer
        if "betas" in optimizer.defaults:
            betas = optimizer.defaults["betas"]
            beta1, beta2 = betas[0], betas[1]
    
    # Format learning rate and weight decay
    # lr=1e-3 -> "1e-03", lr=5e-4 -> "5e-04"
    lr_exp = f"{lr:.0e}".replace("e-0", "e-").replace("e+", "e").replace("+", "")
    if "e-" not in lr_exp and "e" in lr_exp:
        # Handle positive exponents: "1e+00" -> "1e00"
        lr_exp = lr_exp.replace("e+", "e")
    
    # wd=0 -> "0e0", wd=1e-5 -> "1e-05"
    if wd == 0.0:
        wd_str = "0e0"
    else:
        wd_str = f"{wd:.0e}".replace("e-0", "e-").replace("e+", "e").replace("+", "")
    
    # Format betas: 0.9 -> "09", 0.999 -> "0999"
    beta1_str = f"{beta1:.3f}".replace("0.", "")[:2]  # "0.9" -> "09"
    beta2_str = f"{beta2:.3f}".replace("0.", "")  # "0.999" -> "0999"
    
    # Build name
    exp_name = f"hgnn_f{clip_embed_dim}_w{hgnn_hidden}_h{attn_heads}_e{final_embedding_dim}_dr{int(dropout*100)}_{opt_name}_lr{lr_exp}_wd{wd_str}_b{beta1_str}_{beta2_str}"
    return exp_name

def save_model_checkpoint(model, history, best_metrics, best_val_loss, 
                          checkpoint_dir, optimizer=None, exp_name=None):
    """
    Automatically save model checkpoint, training history, metrics, and summary.
    
    Parameters:
    -----------
    model : torch.nn.Module
        Trained model
    history : dict
        Training history with keys: train_loss, val_loss, mse, roc_auc, f1, accuracy
    best_metrics : dict
        Best validation metrics
    best_val_loss : float
        Best validation loss achieved
    checkpoint_dir : Path or str
        Base checkpoint directory
    optimizer : torch.optim.Optimizer, optional
        Optimizer (extracted for learning rate if provided)
    exp_name : str, optional
        Experiment name; if None, auto-generated from model config
    """
    from pathlib import Path
    
    # Auto-generate experiment name if not provided
    if exp_name is None:
        # Extract model parameters
        clip_embed_dim = getattr(model, 'clip_embed_dim', 512)
        attn_heads = getattr(model, 'attn_heads', 4)
        final_embedding_dim = getattr(model, 'final_embedding_dim', 256)
        dropout = getattr(model, 'dropout', 0.2)
        
        lr = 1e-3
        wd = 0.0
        if optimizer is not None:
            # Extract LR and weight decay from optimizer
            for param_group in optimizer.param_groups:
                lr = param_group['lr']
                wd = param_group.get('weight_decay', 0.0)
                break
        
        exp_name = generate_experiment_name(model, optimizer, clip_embed_dim=clip_embed_dim,
                                           attn_heads=attn_heads, final_embedding_dim=final_embedding_dim,
                                           dropout=dropout, lr=lr, wd=wd)
    
    # Create experiment directory
    exp_dir = Path(checkpoint_dir) / exp_name
    exp_dir.mkdir(parents=True, exist_ok=True)
    
    # Save model checkpoint
    torch.save(model.state_dict(), exp_dir / "model.pt")
    print(f"✓ Model checkpoint saved: {exp_dir / 'model.pt'}")
    
    # Save training history
    with open(exp_dir / "history.json", "w") as f:
        history_json = {k: [float(v) for v in vals] for k, vals in history.items()}
        json.dump(history_json, f, indent=2)
    print(f"✓ Training history saved: {exp_dir / 'history.json'}")
    
    # Save best metrics
    with open(exp_dir / "metrics.json", "w") as f:
        metrics_json = {k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                       for k, v in best_metrics.items()}
        json.dump(metrics_json, f, indent=2)
    print(f"✓ Best metrics saved: {exp_dir / 'metrics.json'}")
    
    # Extract model config
    model_config = {}
    if hasattr(model, 'config'):
        model_config = vars(model.config) if hasattr(model.config, '__dict__') else model.config
    else:
        # Try to extract from model parameters
        if hasattr(model, 'hidden_dim'):
            model_config['hidden_dim'] = model.hidden_dim
        if hasattr(model, 'num_heads'):
            model_config['num_heads'] = model.num_heads
        if hasattr(model, 'dropout'):
            model_config['dropout'] = model.dropout
        if hasattr(model, 'num_layers'):
            model_config['num_layers'] = model.num_layers
    
    # Extract optimizer config
    optimizer_config = {}
    if optimizer is not None:
        optimizer_config['type'] = optimizer.__class__.__name__
        optimizer_config['lr'] = optimizer.defaults.get('lr', 'unknown')
        optimizer_config['weight_decay'] = optimizer.defaults.get('weight_decay', 'unknown')
        optimizer_config['betas'] = optimizer.defaults.get('betas', 'unknown')
    
    # Save experiment summary
    summary = {
        "name": exp_name,
        "best_val_loss": float(best_val_loss),
        "best_metrics": {k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                        for k, v in best_metrics.items()},
        "model_config": model_config,
        "optimizer_config": optimizer_config
    }
    
    with open(exp_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(f"✓ Summary saved: {exp_dir / 'summary.json'}")
    
    print(f"\n{'='*70}")
    print(f"Experiment saved to: {exp_dir}")
    print(f"{'='*70}")
    
    return exp_dir

In [ ]:
# Save model checkpoint using the save_model_checkpoint function
model_config = {
    "clip_embed_dim": 512,
    "attr_embed_dim": 256,
    "fusion_hidden": 512,
    "hgnn_hidden": [256, 256, 256],
    "final_embedding_dim": 256,
    "dropout": 0.2,
    "attn_heads": 4,
    "use_hgnn": True
}

optimizer_config = {
    "optim": "adamw",
    "lr": 5e-4,
    "weight_decay": 1e-5
}

training_config = {
    "epochs": 50,
    "batch_size": 32,
    "early_stop_patience": 10
}

exp_dir = save_model_checkpoint(
    model=model,
    history=history,
    best_metrics=best_metrics,
    best_val_loss=best_val_loss,
    checkpoint_dir=CHECKPOINTS_PATH,
    exp_name="hgnn_local_v1",
    model_config=model_config,
    optimizer_config=optimizer_config,
    training_config=training_config
)

## 9. Visualize Training Results

In [ ]:
import matplotlib.pyplot as plt

# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(history['train_loss'], label='Train', marker='o', markersize=3)
axes[0, 0].plot(history['val_loss'], label='Val', marker='s', markersize=3)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training and Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# MSE
axes[0, 1].plot(history['mse'], marker='o', color='orange', markersize=3)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('MSE')
axes[0, 1].set_title('Mean Squared Error')
axes[0, 1].grid(True, alpha=0.3)

# ROC-AUC
axes[1, 0].plot(history['roc_auc'], marker='o', color='green', markersize=3)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('ROC-AUC')
axes[1, 0].set_title('ROC-AUC Score')
axes[1, 0].grid(True, alpha=0.3)

# F1 Score
axes[1, 1].plot(history['f1'], marker='o', color='red', markersize=3)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('F1 Score')
axes[1, 1].set_title('F1 Score')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(exp_dir / "training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

print("Training curves saved to", exp_dir / "training_curves.png")

# Print summary table
print("\n" + "="*60)
print("FINAL METRICS SUMMARY")
print("="*60)
for metric, value in best_metrics.items():
    if isinstance(value, float):
        print(f"{metric:20s}: {value:8.4f}")
    else:
        print(f"{metric:20s}: {value}")
print("="*60)

## 10. Experiment Configuration and Batch Training

In [ ]:
def build_model_from_cfg(cfg: Dict[str, Any], device: str):
    """Build FashionHyperGraphModel from config dict."""
    hgnn_hidden_list = cfg.get("hgnn_hidden_list", [(256, 256, 256)])
    model = FashionHyperGraphModel(
        clip_embed_dim=int(cfg.get("clip_embed_dim", 512)),
        attr_embed_dim=int(cfg.get("attr_embed_dim", 256)),
        fusion_hidden=int(cfg.get("fusion_hidden", 512)),
        final_embedding_dim=int(cfg.get("final_embedding_dim", 256)),
        dropout=float(cfg.get("dropout", 0.2)),
        attn_heads=int(cfg.get("attn_heads", 4)),
        use_hgnn=bool(cfg.get("use_hgnn", True)),
        hgnn_hidden_list=hgnn_hidden_list,
        use_cross_attention=bool(cfg.get("use_cross_attention", False)),
        use_hierarchical_pooling=bool(cfg.get("use_hierarchical_pooling", False)),
        use_moe=bool(cfg.get("use_moe", False)),
        num_experts=int(cfg.get("num_experts", 1))
    )
    return model.to(device)

def make_optimizer(model: torch.nn.Module, cfg: Dict[str, Any]):
    """Create optimizer from config."""
    name = cfg.get("optim", "adam").lower()
    lr = float(cfg.get("lr", 1e-3))
    wd = float(cfg.get("weight_decay", 0.0))
    
    if name == "adam":
        betas = tuple(cfg.get("betas", (0.9, 0.999)))
        return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd, betas=betas)
    elif name == "adamw":
        betas = tuple(cfg.get("betas", (0.9, 0.999)))
        return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, betas=betas)
    elif name == "sgd":
        momentum = float(cfg.get("momentum", 0.0))
        return torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=wd)
    else:
        raise ValueError(f"Unknown optimizer: {name}")

def run_experiment(exp: Dict[str, Any], train_loader, val_loader, 
                   H, Dv_inv_sqrt, De_inv, out_root: Path = None, print_every: int = 10):
    """Run a single experiment with outfit score prediction model."""
    if out_root is None:
        out_root = CHECKPOINTS_PATH
    
    name = exp.get("name", "unknown")
    out_dir = out_root / name
    out_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n{'='*70}")
    print(f"Running experiment: {name}")
    print(f"Description: {exp.get('description', 'N/A')}")
    print(f"{'='*70}")
    
    seed = int(exp.get("seed", 42))
    set_seed(seed)

    # Extract configuration
    model_cfg = exp.get("model_cfg", {})
    optim_cfg = exp.get("optimizer_cfg", {})
    batch_size = int(exp.get("batch_size", 32))
    epochs = int(exp.get("epochs", 30))
    early_stop_patience = int(exp.get("early_stop_patience", 10))

    # Build model and optimizer
    hgnn_hidden_list = model_cfg.get("hgnn_hidden_list", [(256, 256, 256)])
    model = FashionHyperGraphModel(
        clip_embed_dim=int(model_cfg.get("clip_embed_dim", 512)),
        attr_embed_dim=int(model_cfg.get("attr_embed_dim", 256)),
        fusion_hidden=int(model_cfg.get("fusion_hidden", 512)),
        final_embedding_dim=int(model_cfg.get("final_embedding_dim", 256)),
        dropout=float(model_cfg.get("dropout", 0.2)),
        attn_heads=int(model_cfg.get("attn_heads", 4)),
        use_hgnn=bool(model_cfg.get("use_hgnn", True)),
        hgnn_hidden_list=hgnn_hidden_list,
        use_cross_attention=bool(model_cfg.get("use_cross_attention", False)),
        use_hierarchical_pooling=bool(model_cfg.get("use_hierarchical_pooling", False)),
        use_moe=bool(model_cfg.get("use_moe", False)),
        num_experts=int(model_cfg.get("num_experts", 1))
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(optim_cfg.get("lr", 5e-4)),
        weight_decay=float(optim_cfg.get("weight_decay", 1e-5))
    )

    # Rebuild loaders with new batch size if needed
    if batch_size != 32:
        _, _, train_loader_exp, val_loader_exp = build_loaders_from_dataframes(
            clothes_df=df_train_items,
            sets_df=df_train_outfits,
            outfits_col="node_ids",
            label_col="label",
            img_col="img_embedding",
            attr_col="attr_embedding",
            node_id_col="node_id",
            val_split=0.1,
            pad_node_id=0,
            batch_size=batch_size,
            device=device,
            random_state=42
        )
    else:
        train_loader_exp = train_loader
        val_loader_exp = val_loader

    print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Training: {epochs} epochs, batch_size={batch_size}, lr={optim_cfg.get('lr')}\n")


    # Train
    model, best_val_loss, best_metrics, history = train_outfit_regressor_hybrid(
        model=model,
        optimizer=optimizer,
        train_loader=train_loader_exp,
        val_loader=val_loader_exp,
        H=H,
        Dv_inv_sqrt=Dv_inv_sqrt,
        De_inv=De_inv,
        device=device,
        epochs=epochs,
        threshold=0.5,
        early_stop_patience=early_stop_patience,
        early_stop_min_delta=1e-4
    )

    # Save artifacts
    torch.save(model.state_dict(), out_dir / "model.pt")
    
    # Convert to serializable format
    history_json = {k: [float(v) for v in vals] for k, vals in history.items()}
    with open(out_dir / "history.json", "w") as f:
        json.dump(history_json, f, indent=2)
    
    metrics_json = {k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                   for k, v in best_metrics.items()}
    with open(out_dir / "metrics.json", "w") as f:
        json.dump(metrics_json, f, indent=2)

    # Summary
    summary = {
        "name": name,
        "description": exp.get("description", "N/A"),
        "seed": seed,
        "best_val_loss": float(best_val_loss),
        "best_metrics": metrics_json,
        "model_cfg": model_cfg,
        "optimizer_cfg": optim_cfg,
        "training_config": {
            "batch_size": batch_size,
            "epochs": epochs,
            "early_stop_patience": early_stop_patience
        }
    }
    with open(out_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    
    print(f"✅ Exp {name} completed!")
    print(f"   Best Val Loss: {best_val_loss:.6f}")
    print(f"   Best Metrics: F1={best_metrics.get('f1', 0):.4f}, "
          f"Acc={best_metrics.get('accuracy', 0):.4f}, "
          f"ROC-AUC={best_metrics.get('roc_auc', 0):.4f}")
    print(f"   Saved to: {out_dir}")
    
    return summary

def run_all_experiments(experiments: Dict[str, Dict[str, Any]], train_loader, val_loader, 
                       H, Dv_inv_sqrt, De_inv, out_root: Path = None):
    """Run all experiments and create master summary."""
    if out_root is None:
        out_root = CHECKPOINTS_PATH
    
    out_root.mkdir(parents=True, exist_ok=True)
    results = []
    
    for key, exp in experiments.items():
        if "name" not in exp:
            exp["name"] = key
        try:
            res = run_experiment(exp, train_loader, val_loader, H, Dv_inv_sqrt, De_inv,
                               out_root=out_root, print_every=10)
            results.append(res)
        except Exception as e:
            print(f"❌ Experiment {exp.get('name', '?')} failed: {e}")
            import traceback
            traceback.print_exc()
            results.append({"name": exp.get("name", "?"), "error": str(e)})

    # Save master summary
    with open(out_root / "master_summary.json", "w") as f:
        json.dump(results, f, indent=2)

    # Create CSV summary
    csv_path = out_root / "experiments_summary.csv"
    import csv
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["name", "best_val_loss", "best_mse", "roc_auc", "f1", "status"])
        for r in results:
            if "error" in r:
                writer.writerow([r.get("name"), "ERROR", "ERROR", "ERROR", "ERROR", r.get("error")])
            else:
                best_metrics = r.get("best_metrics", {})
                writer.writerow([
                    r.get("name"),
                    round(r.get("best_val_loss", 0), 6),
                    round(best_metrics.get("mse", 0), 4),
                    round(best_metrics.get("roc_auc", 0), 4),
                    round(best_metrics.get("f1", 0), 4),
                    "OK"
                ])

    print(f"\n{'='*70}")
    print(f"All experiments completed!")
    print(f"Master summary: {out_root / 'master_summary.json'}")
    print(f"CSV summary: {csv_path}")
    print(f"{'='*70}")
    
    return results

print("Experiment utilities defined")

### Define Experiments

In [ ]:
# Define 10 experiment configurations for outfit score prediction
# Names will be automatically generated based on model and optimizer config

def gen_exp_name(model_cfg, optimizer_cfg):
    """Generate experiment name from config.
    
    Format: hgnn_f{clip}_w{hgnn}_h{heads}_e{embed}_dr{drop}_ca{CA}_hp{HP}_moe{MoE}_{opt}_lr{lr}_wd{wd}
    """
    clip_dim = model_cfg["clip_embed_dim"]
    use_hgnn = model_cfg["use_hgnn"]
    embed_dim = model_cfg["final_embedding_dim"]
    dropout_int = int(model_cfg["dropout"] * 100)
    
    opt_name = optimizer_cfg["optim"].lower()
    lr = optimizer_cfg["lr"]
    wd = optimizer_cfg["weight_decay"]
    
    # Format lr and wd more compactly
    if lr == 5e-4:
        lr_str = "5e4"
    elif lr == 3e-4:
        lr_str = "3e4"
    elif lr == 1e-4:
        lr_str = "1e4"
    else:
        lr_str = f"{lr:.0e}".replace("e-0", "e").replace("e-", "e")
    
    if wd == 1e-5:
        wd_str = "1e5"
    elif wd == 5e-4:
        wd_str = "5e4"
    elif wd == 0:
        wd_str = "0"
    else:
        wd_str = f"{wd:.0e}".replace("e-0", "e").replace("e-", "e")
    
    # Advanced features flags
    ca = "T" if model_cfg.get("use_cross_attention", False) else "F"
    hp = "T" if model_cfg.get("use_hierarchical_pooling", False) else "F"
    moe_flag = model_cfg.get("use_moe", False)
    num_experts = model_cfg.get("num_experts", 1)
    moe_str = f"{num_experts}" if moe_flag and num_experts > 1 else "0"
    
    if use_hgnn:
        # Get HGNN architecture info
        if "hgnn_hidden_list" in model_cfg and model_cfg["hgnn_hidden_list"]:
            # Format all branches
            branch_strs = []
            for branch in model_cfg["hgnn_hidden_list"]:
                branch_str = "x".join(str(h) for h in branch)
                branch_strs.append(branch_str)
            hgnn_hidden = "_".join(branch_strs) if len(branch_strs) > 1 else branch_strs[0]
        else:
            hgnn_hidden = "x".join(str(h) for h in model_cfg.get("hgnn_hidden", (256,)))
        
        heads = model_cfg["attn_heads"]
        # hgnn_f{clip}_w{hgnn}_h{heads}_e{embed}_dr{drop}_ca{ca}_hp{hp}_moe{moe}_{opt}_lr{lr}_wd{wd}
        return f"hgnn_f{clip_dim}_w{hgnn_hidden}_h{heads}_e{embed_dim}_dr{dropout_int}_ca{ca}_hp{hp}_moe{moe_str}_{opt_name}_lr{lr_str}_wd{wd_str}"
    else:
        # baseline_f{clip}_e{embed}_dr{drop}_ca{ca}_hp{hp}_moe{moe}_{opt}_lr{lr}_wd{wd}
        return f"baseline_f{clip_dim}_e{embed_dim}_dr{dropout_int}_ca{ca}_hp{hp}_moe{moe_str}_{opt_name}_lr{lr_str}_wd{wd_str}"


# Define experiment configurations
exp_configs = [
    
    # ===== Group 2: Dual-branch (memory manageable) =====
    {
        "description": "Dual-branch: (256,256,256)+(128) - asymmetric",
        "model_cfg": {
            "clip_embed_dim": 512, "attr_embed_dim": 256,
            "fusion_hidden": 512, "hgnn_hidden_list": [(128,128)],
            "final_embedding_dim": 256, "dropout": 0.1,
            "attn_heads": 4, "use_hgnn": True,
            "use_cross_attention": False, "use_hierarchical_pooling": True,
            "use_moe": False, "num_experts": 1
        },
        "optimizer_cfg": {"optim": "adamw", "lr": 1e-4, "weight_decay": 1e-5},
        "batch_size": 32, "epochs": 100, "early_stop_patience": 3
    },
        {
        "description": "Dual-branch: (256,256,256)+(128) - asymmetric",
        "model_cfg": {
            "clip_embed_dim": 512, "attr_embed_dim": 256,
            "fusion_hidden": 512, "hgnn_hidden_list": [(128,128)],
            "final_embedding_dim": 256, "dropout": 0.25,
            "attn_heads": 4, "use_hgnn": True,
            "use_cross_attention": False, "use_hierarchical_pooling": True,
            "use_moe": False, "num_experts": 1
        },
        "optimizer_cfg": {"optim": "adamw", "lr": 1e-4, "weight_decay": 1e-5},
        "batch_size": 32, "epochs": 100, "early_stop_patience": 3
    },
                {
        "description": "Dual-branch: (256,256,256)+(128) - asymmetric",
        "model_cfg": {
            "clip_embed_dim": 512, "attr_embed_dim": 256,
            "fusion_hidden": 512, "hgnn_hidden_list": [(128,128)],
            "final_embedding_dim": 256, "dropout": 0.5,
            "attn_heads": 4, "use_hgnn": True,
            "use_cross_attention": False, "use_hierarchical_pooling": True,
            "use_moe": False, "num_experts": 1
        },
        "optimizer_cfg": {"optim": "adamw", "lr": 1e-4, "weight_decay": 1e-5},
        "batch_size": 32, "epochs": 100, "early_stop_patience": 3
    },
]



# Build EXPERIMENTS dict with auto-generated names
EXPERIMENTS = {}
for cfg in exp_configs:
    exp_name = gen_exp_name(cfg["model_cfg"], cfg["optimizer_cfg"])
    EXPERIMENTS[exp_name] = {
        "name": exp_name,
        "description": cfg["description"],
        "model_cfg": cfg["model_cfg"],
        "optimizer_cfg": cfg["optimizer_cfg"],
        "batch_size": cfg["batch_size"],
        "epochs": cfg["epochs"],
        "early_stop_patience": cfg["early_stop_patience"]
    }

print("\n" + "="*80)
print("OUTFIT SCORE PREDICTION: 10 EXPERIMENTS")
print("="*80)
print(f"\nTotal experiments: {len(EXPERIMENTS)}\n")
for idx, (key, cfg) in enumerate(EXPERIMENTS.items(), 1):
    print(f"{idx:2d}. {cfg['name']:60s} - {cfg['description']}")
    # Handle both hgnn_hidden_list and hgnn_hidden
    if "hgnn_hidden_list" in cfg['model_cfg']:
        layers_info = f"HGNN list: {cfg['model_cfg']['hgnn_hidden_list']}"
    else:
        layers_info = f"HGNN tuple: {cfg['model_cfg']['hgnn_hidden']}"
    print(f"    {layers_info}, "
          f"Dropout: {cfg['model_cfg']['dropout']}, LR: {cfg['optimizer_cfg']['lr']}, "
          f"Batch: {cfg['batch_size']}, Epochs: {cfg['epochs']}")
    print()

### Run All Experiments

In [ ]:
# Run all experiments
results = run_all_experiments(
    experiments=EXPERIMENTS,
    train_loader=train_loader,
    val_loader=val_loader,
    H=H,
    Dv_inv_sqrt=Dv_inv_sqrt,
    De_inv=De_inv,
    out_root=CHECKPOINTS_PATH
)

# Display summary
print("\nExperiment Results Summary:")
print("-" * 80)
import pandas as pd
df_results = pd.read_csv(CHECKPOINTS_PATH / "experiments_summary.csv")
print(df_results.to_string(index=False))
print("-" * 80)

## 11. Results Analysis and Visualization

In [1]:
# ============================================
# Fashion Classification Model: Metrics Analysis Function
# ============================================

import json
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def analyze_classification_models(parent_folder: str):
    """
    Generate comprehensive metrics graphs and summaries for classification models.
    
    Args:
        parent_folder: Path to parent folder containing experiment subdirectories
    
    Returns:
        dict: all_experiments with loaded data from all folders
    
    Generates files in 'parent_folder/comparison/' folder:
        - training_curves.html & .png: Train/Val loss across experiments
        - metrics_comparison.html & .png: Overall metrics (F1, ROC-AUC, Precision, Recall, etc)
        - val_loss_ranking.html & .png: Model ranking by validation loss
        - classification_metrics_summary.csv: Summary table of all metrics
    """
    
    parent_dir = Path(parent_folder)
    outputs_dir = parent_dir / "comparison"
    outputs_dir.mkdir(parents=True, exist_ok=True)
    
    exp_dirs = sorted([d for d in parent_dir.iterdir() if d.is_dir() and d.name != "comparison"])
    
    if not exp_dirs:
        print(f"❌ No experiment folders found in {parent_folder}")
        return {}
    
    print(f"Found {len(exp_dirs)} experiments:")
    for d in exp_dirs:
        print(f"  - {d.name}")
    
    # Load all experiment data
    all_experiments = {}
    for exp_dir in exp_dirs:
        try:
            with open(exp_dir / "history.json") as f:
                history = json.load(f)
            with open(exp_dir / "metrics.json") as f:
                metrics = json.load(f)
            with open(exp_dir / "summary.json") as f:
                summary = json.load(f)
            all_experiments[exp_dir.name] = {
                "history": history,
                "metrics": metrics,
                "summary": summary
            }
        except Exception as e:
            print(f"  ⚠️  Error loading {exp_dir.name}: {e}")
    
    print(f"\n✅ Loaded {len(all_experiments)} experiments successfully\n")
    
    # ===== Plot training curves =====
    print("📊 Generating training curves...")
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Training Loss", "Validation Loss"),
        specs=[[{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    for exp_name, data in all_experiments.items():
        hist = data["history"]
        epochs = list(range(1, len(hist["train_loss"]) + 1))
        
        fig.add_trace(
            go.Scatter(
                x=epochs,
                y=hist["train_loss"],
                mode="lines",
                name=f"{exp_name} (train)",
                line=dict(width=2)
            ),
            row=1, col=1
        )
        
        if "val_loss" in hist:
            fig.add_trace(
                go.Scatter(
                    x=epochs,
                    y=hist["val_loss"],
                    mode="lines",
                    name=f"{exp_name} (val)",
                    line=dict(width=2, dash="dash")
                ),
                row=1, col=2
            )
    
    fig.update_xaxes(title_text="Epoch", row=1, col=1)
    fig.update_xaxes(title_text="Epoch", row=1, col=2)
    fig.update_yaxes(title_text="Loss", row=1, col=1)
    fig.update_yaxes(title_text="Loss", row=1, col=2)
    fig.update_layout(height=500, width=1400, title_text="Training Curves Across Experiments")
    fig.write_html(str(outputs_dir / "training_curves.html"))
    fig.write_image(str(outputs_dir / "training_curves.png"), width=1400, height=500)
    fig.write_image(str(outputs_dir / "training_curves.jpg"), width=1400, height=500)
    fig.show()
    print("  ✅ Saved: training_curves.html, training_curves.png, training_curves.jpg")
    
    # ===== Metrics comparison =====
    print("\n📊 Generating overall metrics comparison...")
    fig = go.Figure()
    
    metric_names = ["F1", "ROC-AUC", "Precision", "Recall", "Accuracy", "MSE"]
    
    for exp_name, data in all_experiments.items():
        metrics = data["metrics"]
        
        f1 = metrics.get("f1", 0)
        roc_auc = metrics.get("roc_auc", 0)
        precision = metrics.get("precision", 0)
        recall = metrics.get("recall", 0)
        accuracy = metrics.get("accuracy", 0)
        mse = metrics.get("mse", 0)
        
        fig.add_trace(go.Bar(
            name=exp_name,
            x=metric_names,
            y=[f1, roc_auc, precision, recall, accuracy, mse],
            text=[f"{v:.3f}" for v in [f1, roc_auc, precision, recall, accuracy, mse]],
            textposition="auto"
        ))
    
    fig.update_layout(
        title="Overall Classification Metrics Comparison",
        barmode="group",
        height=500,
        width=1200,
        yaxis_title="Score",
        yaxis=dict(range=[0, 1.05])
    )
    fig.write_html(str(outputs_dir / "metrics_comparison.html"))
    fig.write_image(str(outputs_dir / "metrics_comparison.png"), width=1200, height=500)
    fig.write_image(str(outputs_dir / "metrics_comparison.jpg"), width=1200, height=500)
    fig.show()
    print("  ✅ Saved: metrics_comparison.html, metrics_comparison.png, metrics_comparison.jpg")
    
    # ===== Create comprehensive summary table =====
    print("\n📊 Creating summary table...")
    summary_rows = []
    for exp_name, data in all_experiments.items():
        metrics = data["metrics"]
        
        row = {"experiment": exp_name}
        row["val_loss"] = metrics.get("val_loss", 0)
        row["mse"] = metrics.get("mse", 0)
        row["roc_auc"] = metrics.get("roc_auc", 0)
        row["f1"] = metrics.get("f1", 0)
        row["accuracy"] = metrics.get("accuracy", 0)
        row["precision"] = metrics.get("precision", 0)
        row["recall"] = metrics.get("recall", 0)
        row["avg_score"] = metrics.get("avg_score", 0)
        row["tp"] = metrics.get("tp", 0)
        row["fp"] = metrics.get("fp", 0)
        row["tn"] = metrics.get("tn", 0)
        row["fn"] = metrics.get("fn", 0)
        
        summary_rows.append(row)
    
    summary_df = pd.DataFrame(summary_rows)
    summary_df = summary_df.round(4)
    
    print("\n" + "="*160)
    print("COMPREHENSIVE METRICS SUMMARY")
    print("="*160)
    print(summary_df.to_string(index=False))
    print("="*160)
    
    summary_df.to_csv(str(outputs_dir / "classification_metrics_summary.csv"), index=False)
    print("  ✅ Saved: classification_metrics_summary.csv")
    
    # ===== Validation Loss Ranking =====
    print("\n📊 Generating validation loss ranking...")
    val_loss_data = [(exp_name, all_experiments[exp_name]["metrics"].get("val_loss", float('inf'))) 
                      for exp_name in all_experiments.keys()]
    val_loss_data.sort(key=lambda x: x[1])
    
    fig_loss = go.Figure(go.Bar(
        y=[name for name, _ in val_loss_data],
        x=[loss for _, loss in val_loss_data],
        orientation='h',
        text=[f"{loss:.4f}" for _, loss in val_loss_data],
        textposition='auto',
        marker=dict(
            color=[loss for _, loss in val_loss_data],
            colorscale='RdYlGn_r',
            showscale=True
        )
    ))
    
    fig_loss.update_layout(
        title="Validation Loss: Model Ranking",
        xaxis_title="Validation Loss (Lower is Better)",
        height=400,
        width=900,
        showlegend=False
    )
    fig_loss.write_html(str(outputs_dir / "val_loss_ranking.html"))
    fig_loss.write_image(str(outputs_dir / "val_loss_ranking.png"), width=900, height=400)
    fig_loss.write_image(str(outputs_dir / "val_loss_ranking.jpg"), width=900, height=400)
    fig_loss.show()
    print("  ✅ Saved: val_loss_ranking.html, val_loss_ranking.png, val_loss_ranking.jpg")
    
    # ===== Average metrics summary =====
    print("\n" + "="*100)
    print("AVERAGE METRICS ACROSS ALL MODELS")
    print("="*100)
    
    avg_f1 = np.mean([all_experiments[e]["metrics"].get("f1", 0) for e in all_experiments.keys()])
    avg_roc_auc = np.mean([all_experiments[e]["metrics"].get("roc_auc", 0) for e in all_experiments.keys()])
    avg_precision = np.mean([all_experiments[e]["metrics"].get("precision", 0) for e in all_experiments.keys()])
    avg_recall = np.mean([all_experiments[e]["metrics"].get("recall", 0) for e in all_experiments.keys()])
    avg_accuracy = np.mean([all_experiments[e]["metrics"].get("accuracy", 0) for e in all_experiments.keys()])
    avg_mse = np.mean([all_experiments[e]["metrics"].get("mse", 0) for e in all_experiments.keys()])
    avg_val_loss = np.mean([all_experiments[e]["metrics"].get("val_loss", 0) for e in all_experiments.keys()])
    
    print(f"F1 Score: {avg_f1:.4f} | ROC-AUC: {avg_roc_auc:.4f} | Precision: {avg_precision:.4f}")
    print(f"Recall: {avg_recall:.4f} | Accuracy: {avg_accuracy:.4f}")
    print(f"MSE: {avg_mse:.4f} | Validation Loss: {avg_val_loss:.4f}")
    print("="*100)
    
    return all_experiments

# Example usage:
all_experiments = analyze_classification_models(r"C:\TFM\APP\ml_pipeline\experiments_fashion_score")

Found 38 experiments:
  - baseline_f512_e128_dr10_adamw_lr5e-4_wd1e-5_b9_999
  - baseline_f512_e128_dr30_adamw_lr5e-4_wd1e-4_b9_999
  - hgnn_f512_w128_128_128_128_h4_e256_dr20_caF_hpF_moe0_adamw_lr3e4_wd1e5
  - hgnn_f512_w128_128_128_h2_e256_dr10_caF_hpT_moe0_adamw_lr1e4_wd1e5
  - hgnn_f512_w128x128_128x128_h4_e256_dr20_caF_hpF_moe0_adamw_lr3e4_wd1e5
  - hgnn_f512_w128x128_h2_e128_dr10_adamw_lr5e-4_wd1e-5_b9_999
  - hgnn_f512_w128x128_h3_e256_dr15_caF_hpT_moe0_adamw_lr1e4_wd1e5
  - hgnn_f512_w128x128_h4_e256_dr10_caF_hpT_moe0_adamw_lr1e4_wd1e5
  - hgnn_f512_w128x128_h4_e256_dr15_caF_hpT_moe0_adamw_lr1e4_wd1e5
  - hgnn_f512_w128x128_h4_e256_dr15_caF_hpT_moe2_adamw_lr1e4_wd1e5
  - hgnn_f512_w128x128_h4_e256_dr15_caF_hpT_moe3_adamw_lr1e4_wd1e5
  - hgnn_f512_w128x128_h4_e256_dr25_caF_hpT_moe0_adamw_lr1e4_wd1e5
  - hgnn_f512_w128x128_h4_e256_dr50_caF_hpT_moe0_adamw_lr1e4_wd1e5
  - hgnn_f512_w128x128_h5_e256_dr15_caF_hpT_moe0_adamw_lr1e4_wd1e5
  - hgnn_f512_w128x128x128_h2_e128_dr15_adamw_lr

  ✅ Saved: training_curves.html, training_curves.png, training_curves.jpg

📊 Generating overall metrics comparison...


  ✅ Saved: metrics_comparison.html, metrics_comparison.png, metrics_comparison.jpg

📊 Creating summary table...

COMPREHENSIVE METRICS SUMMARY
                                                             experiment  val_loss    mse  roc_auc     f1  accuracy  precision  recall  avg_score   tp   fp   tn   fn
                     baseline_f512_e128_dr10_adamw_lr5e-4_wd1e-5_b9_999    0.5386 0.1813   0.8056 0.7262    0.7205     0.7115  0.7416     0.5262 1260  511 1189  439
                     baseline_f512_e128_dr30_adamw_lr5e-4_wd1e-4_b9_999    0.5407 0.1820   0.8034 0.7074    0.7196     0.7394  0.6780     0.4989 1152  406 1294  547
 hgnn_f512_w128_128_128_128_h4_e256_dr20_caF_hpF_moe0_adamw_lr3e4_wd1e5    0.5129 0.1717   0.8249 0.7209    0.7367     0.7666  0.6804     0.4720 1156  352 1348  543
     hgnn_f512_w128_128_128_h2_e256_dr10_caF_hpT_moe0_adamw_lr1e4_wd1e5    0.4338 0.1368   0.8886 0.8036    0.8085     0.8243  0.7840     0.4706 1332  284 1416  367
 hgnn_f512_w128x128_128x128_h4_e

  ✅ Saved: val_loss_ranking.html, val_loss_ranking.png, val_loss_ranking.jpg

AVERAGE METRICS ACROSS ALL MODELS
F1 Score: 0.7478 | ROC-AUC: 0.8289 | Precision: 0.7437
Recall: 0.7595 | Accuracy: 0.7577
MSE: 0.1628 | Validation Loss: 0.4995


## 12. Model Loading and Inference

In [ ]:
# Load test data using load_and_process_data function
print("\n" + "="*70)
print("LOADING TEST DATA")
print("="*70)

df_test_outfits, df_test_items, img2id_test = load_and_process_data(
    TEST_OUTFITS_PATH, TEST_ITEMS_PATH, "test"
)

# Generate attribute embeddings for test items
print("\nGenerating attribute embeddings for test items...")
attribute_embeddings_test, _ = generate_attribute_embeddings(
    df_test_items, device=device
)
df_test_items["Xa"] = list(attribute_embeddings_test)
print(f"✓ Test attribute embeddings generated: shape {attribute_embeddings_test.shape}")

# Build hyperedges for test data
print("\nBuilding hyperedges for test data...")
edge_list_test = build_hyperedges_from_outfits(df_test_outfits, wardrobe_name="test", outfits_col="node_ids")
print(f"✓ Test hyperedges built: {len(edge_list_test)} hyperedges")
print(f"  Sample hyperedges (first 3): {edge_list_test[:3]}")

# Build incidence matrix and degree matrices for test
print("\nBuilding incidence matrix for test data...")
N_nodes_test = len(df_test_items)
H_test, Dv_inv_sqrt_test, De_inv_test = build_incidence_matrix(
    num_nodes=N_nodes_test,
    hyperedges=edge_list_test,
    device=device
)
print(f"✓ Test incidence matrix built:")
print(f"  Shape: {H_test.shape}")
print(f"  Dv_inv_sqrt shape: {Dv_inv_sqrt_test.shape}")
print(f"  De_inv shape: {De_inv_test.shape}")

print("\n" + "="*70)
print("✅ TEST DATA PREPARATION COMPLETE")
print("="*70)
print(f"Summary:")
print(f"  • Test items: {len(df_test_items)}")
print(f"  • Test outfits: {len(df_test_outfits)}")
print(f"  • Test hyperedges: {len(edge_list_test)}")
print(f"  • Node ID range: 0 to {N_nodes_test-1}")
print(f"  • Attribute embedding dim: {attribute_embeddings_test.shape[1]}")


In [ ]:
# Build test data loaders
print("\n" + "="*70)
print("BUILDING TEST DATASETS AND LOADERS")
print("="*70)

# Build Xc and Xa feature matrices for test items
print("\nBuilding feature matrices for test items...")
Xc_test, Xa_test = build_Xc_Xa_from_clothes_df(
    df_test_items,
    img_col="img_embedding",
    attr_col="Xa",
    node_id_col="node_id"
)
print(f"✓ Feature matrices built:")
print(f"  Xc_test shape: {Xc_test.shape}")
print(f"  Xa_test shape: {Xa_test.shape}")

# Parse test outfit node lists
def _parse_node_list_test(x):
    """Parse node list from various formats."""
    if isinstance(x, (list, tuple, np.ndarray)):
        return [int(i) for i in x]
    try:
        return [int(i) for i in eval(x)]
    except Exception:
        s = str(x).strip("[]")
        if s == "":
            return []
        return [int(i) for i in s.split(",")]

test_node_lists = [_parse_node_list_test(x) for x in df_test_outfits['node_ids'].values]
test_labels = np.array(df_test_outfits['label'].values, dtype=np.float32)

print(f"\n✓ Test outfits parsed:")
print(f"  Total outfits: {len(test_node_lists)}")
print(f"  Labels shape: {test_labels.shape}")

# Create test dataset
print("\nCreating test dataset...")
test_ds = NodeIdPaddedOutfitDataset(
    outfits_node_ids=test_node_lists,
    labels=test_labels,
    Xc=Xc_test,
    Xa=Xa_test,
    pad_node_id=0,
    max_len=max(len(o) for o in test_node_lists),
    device=device
)

# Create test loader
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=padded_collate)

print(f"✓ Test dataset created: {len(test_ds)} samples")
print(f"✓ Test loader created: {len(test_loader)} batches")
print(f"  Batch size: 32")
print(f"  Max outfit length: {test_ds.max_len}")

# Verify test loader
print("\n✓ Verifying test loader...")
test_batch = next(iter(test_loader))
print(f"  Batch nodes shape: {test_batch['nodes'].shape}")
print(f"  Batch mask shape: {test_batch['mask'].shape}")
print(f"  Batch label shape: {test_batch['label'].shape}")
print(f"  Number of batches: {len(test_loader)}")

print("\n" + "="*70)
print("✅ TEST DATA LOADING COMPLETE")
print("="*70)


In [ ]:
def load_model_from_folder(folder: Path, device: str = "cpu"):
    """Load a trained model and its config from experiment folder."""
    summary_path = folder / "summary.json"
    ckpt_path = folder / "model.pt"

    with open(summary_path, "r") as f:
        summary = json.load(f)
    
    model_cfg = summary.get("model_cfg", {})
    
    # Rebuild model
    model = build_model_from_cfg(model_cfg, device=device)
    
    # Load weights
    state_dict = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    
    return model, summary

def predict_outfit_score(model, outfit_node_ids, Xc, Xa, H, Dv_inv_sqrt, De_inv, device="cpu"):
    """Predict compatibility score for a single outfit using integrated model."""
    model.eval()
    
    with torch.no_grad():
        Xc_t = torch.tensor(Xc, dtype=torch.float32, device=device)
        Xa_t = torch.tensor(Xa, dtype=torch.float32, device=device)
        
        # Create batch with single outfit - outfit_node_ids is already a list
        outfit_nodes = torch.tensor([outfit_node_ids], dtype=torch.long, device=device)  # Shape: [1, L]
        outfit_mask = torch.ones(1, len(outfit_node_ids), device=device)  # Shape: [1, L]
        
        # Model returns outfit score directly (already in [0, 1] from sigmoid)
        outfit_scores, _, _, _ = model(Xc_t, Xa_t, H, Dv_inv_sqrt, De_inv,
                                       outfit_nodes=outfit_nodes, outfit_mask=outfit_mask)
        
        score = outfit_scores[0].item()
    
    return score

# Initialize experiment directories
parent_dir = Path(r"C:\TFM\APP\ml_pipeline\experiments_fashion_score")
exp_dirs = sorted([d for d in parent_dir.iterdir() if d.is_dir() and d.name != "comparison"])

# Find best model by validation loss
def get_best_val_loss(exp_dir):
    try:
        with open(exp_dir / "summary.json") as f:
            summary = json.load(f)
        return summary.get("best_val_loss", float('inf'))
    except:
        return float('inf')

# Example: Load best model
best_model_dir = sorted(exp_dirs, key=get_best_val_loss)[0]
print(f"Loading best model from: {best_model_dir.name}")

best_model, best_summary = load_model_from_folder(best_model_dir, device=device)
print(f"✅ Model loaded successfully!")
print(f"Best val loss: {best_summary.get('best_val_loss', 'N/A')}")
print(f"Best metrics: {best_summary.get('best_metrics', {})}")

# Test prediction on a sample outfit
print("\n" + "="*70)
print("Testing inference on sample outfits")
print("="*70)

# Get test data
test_sample_indices = np.random.choice(len(df_test_outfits), 3, replace=False)

for idx in test_sample_indices:
    row = df_test_outfits.iloc[idx]
    outfit_node_ids = row["node_ids"]
    label = row["label"]
    
    if isinstance(outfit_node_ids, str):
        outfit_node_ids = ast.literal_eval(outfit_node_ids)
    
    # Get feature matrices
    Xc = Xc_test
    Xa = Xa_test
    
    # Predict
    score = predict_outfit_score(best_model, outfit_node_ids, Xc, Xa, H_test, Dv_inv_sqrt_test, De_inv_test, device=device)
    
    prediction = "Compatible ✅" if score > 0.5 else "Incompatible ❌"
    label_str = "Compatible" if label == 1 else "Incompatible"
    
    print(f"Outfit {idx}: {len(outfit_node_ids)} items")
    print(f"  Predicted score: {score:.4f} ({prediction})")
    print(f"  True label: {label_str}")
    print()

## 14. Reinforcement Learning - Interactive Preference Learning

In [ ]:
class PreferenceLearner:
    """Learn user preferences through reinforcement learning."""
    
    def __init__(self, model, optimizer, device='cpu'):
        self.model = model
        self.optimizer = optimizer
        self.device = device
        self.feedback_history = []
        self.learning_curves = {'rewards': [], 'loss': []}
    
    def get_outfit_prediction(self, outfit_node_ids, Xc, Xa, H, Dv_inv_sqrt, De_inv):
        """Get model prediction for an outfit using integrated model."""
        self.model.eval()
        with torch.no_grad():
            Xc_t = torch.tensor(Xc, dtype=torch.float32, device=self.device)
            Xa_t = torch.tensor(Xa, dtype=torch.float32, device=self.device)
            
            outfit_nodes = torch.tensor([outfit_node_ids], dtype=torch.long, device=self.device).unsqueeze(0)
            outfit_mask = torch.ones(1, len(outfit_node_ids), device=self.device)
            
            # Model returns outfit score directly (already in [0, 1])
            outfit_scores, _, _, _ = self.model(Xc_t, Xa_t, H, Dv_inv_sqrt, De_inv,
                                               outfit_nodes=outfit_nodes, outfit_mask=outfit_mask)
            
            score = outfit_scores[0].item()
        
        return score
    
    def learn_from_feedback(self, outfit_node_ids, feedback_reward, Xc, Xa, H, Dv_inv_sqrt, De_inv,
                           learning_rate_multiplier=1.0):
        """Update model based on user feedback (reward) using integrated model."""
        self.model.train()
        
        # Convert feedback to target (0 or 1)
        target = torch.tensor([float(feedback_reward)], dtype=torch.float32, device=self.device)
        
        Xc_t = torch.tensor(Xc, dtype=torch.float32, device=self.device)
        Xa_t = torch.tensor(Xa, dtype=torch.float32, device=self.device)
        
        with torch.enable_grad():
            outfit_nodes = torch.tensor([outfit_node_ids], dtype=torch.long, device=self.device).unsqueeze(0)
            outfit_mask = torch.ones(1, len(outfit_node_ids), device=self.device)
            
            # Model returns outfit score directly (already in [0, 1])
            outfit_scores, _, _, _ = self.model(Xc_t, Xa_t, H, Dv_inv_sqrt, De_inv,
                                               outfit_nodes=outfit_nodes, outfit_mask=outfit_mask)
            
            score = outfit_scores[0]
            
            # Binary cross-entropy loss (scores already in [0, 1])
            loss = F.binary_cross_entropy(score.unsqueeze(0), target)
            
            # Scale gradient by learning rate multiplier
            scaled_loss = loss * learning_rate_multiplier
            
            self.optimizer.zero_grad()
            scaled_loss.backward()
            self.optimizer.step()
        
        # Record feedback
        self.feedback_history.append({
            'outfit': outfit_node_ids,
            'reward': feedback_reward,
            'loss': loss.item()
        })
        self.learning_curves['rewards'].append(feedback_reward)
        self.learning_curves['loss'].append(loss.item())
        
        return loss.item()
    
    def batch_learn_from_feedback(self, outfit_node_ids_list, rewards_list, Xc, Xa, 
                                 H, Dv_inv_sqrt, De_inv, num_epochs=5):
        """Learn from multiple feedback samples using integrated model."""
        self.model.train()
        
        Xc_t = torch.tensor(Xc, dtype=torch.float32, device=self.device)
        Xa_t = torch.tensor(Xa, dtype=torch.float32, device=self.device)
        
        total_loss = 0.0
        avg_reward = np.mean(rewards_list)
        
        for epoch in range(num_epochs):
            for outfit_ids, reward in zip(outfit_node_ids_list, rewards_list):
                with torch.enable_grad():
                    outfit_nodes = torch.tensor([outfit_ids], dtype=torch.long, device=self.device).unsqueeze(0)
                    outfit_mask = torch.ones(1, len(outfit_ids), device=self.device)
                    
                    # Model returns outfit score directly
                    outfit_scores, _, _, _ = self.model(Xc_t, Xa_t, H, Dv_inv_sqrt, De_inv,
                                                       outfit_nodes=outfit_nodes, outfit_mask=outfit_mask)
                    
                    score = outfit_scores[0]
                    target = torch.tensor([float(reward)], dtype=torch.float32, device=self.device)
                    
                    loss = F.binary_cross_entropy(score.unsqueeze(0), target)
                    
                    self.optimizer.zero_grad()
                    loss.backward()
                    self.optimizer.step()
                    
                    total_loss += loss.item()
        
        avg_loss = total_loss / (len(outfit_node_ids_list) * num_epochs)
        
        self.learning_curves['rewards'].append(avg_reward)
        self.learning_curves['loss'].append(avg_loss)
        
        return avg_loss


### Interactive Feedback Loop (Non-Jupyter)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def predict_and_show_outfit(model, row, X_clip, X_attr, H, Dv_inv_sqrt, De_inv, device='cpu'):
    """Predict outfit compatibility score and display clothing information."""
    model.eval()
    
    outfit_node_ids = row['node_ids']
    label = row['label']
    
    if isinstance(outfit_node_ids, str):
        outfit_node_ids = ast.literal_eval(outfit_node_ids)
    
    Xc = torch.tensor(X_clip, dtype=torch.float32, device=device)
    Xa = torch.tensor(X_attr, dtype=torch.float32, device=device)
    
    with torch.no_grad():
        # Get outfit embedding and prediction
        outfit_nodes = torch.tensor([outfit_node_ids], dtype=torch.long, device=device)
        outfit_mask = torch.ones(1, len(outfit_node_ids), device=device)
        
        outfit_scores, _, _, _ = model(Xc, Xa, H, Dv_inv_sqrt, De_inv,
                                       outfit_nodes=outfit_nodes, outfit_mask=outfit_mask)
        
        score = outfit_scores[0].item()
    
    # Display outfit information
    print(f"\n{'='*70}")
    print(f"OUTFIT COMPATIBILITY ASSESSMENT")
    print(f"{'='*70}")
    print(f"Outfit ID: {outfit_node_ids}")
    print(f"\n📊 Model Prediction Score: {score:.4f}")
    
    # Interpretation
    if score > 0.8:
        interpretation = "🌟 Very Compatible"
    elif score > 0.6:
        interpretation = "✅ Compatible"
    elif score > 0.4:
        interpretation = "⚠️  Neutral"
    else:
        interpretation = "❌ Incompatible"
    
    print(f"Interpretation: {interpretation}")
    print(f"Ground Truth Label: {'✅ Compatible' if label == 1 else '❌ Incompatible'}")
    print(f"{'='*70}\n")
    
    return score, outfit_node_ids


def interactive_feedback_loop(model, df_outfits, X_clip, X_attr, H, Dv_inv_sqrt, De_inv, device='cpu'):
    """Interactive session to collect feedback on outfit predictions."""
    
    feedback_data = {
        'outfit_id': [],
        'predicted_score': [],
        'user_feedback': [],
        'correct': []
    }
    
    def next_outfit(_=None):
        clear_output(wait=True)
        
        # Select random outfit
        row = df_outfits.iloc[np.random.randint(0, len(df_outfits))]
        score, outfit_node_ids = predict_and_show_outfit(
            model, row, X_clip, X_attr, H, Dv_inv_sqrt, De_inv, device
        )
        
        def handle_feedback(feedback):
            # Record feedback
            feedback_data['outfit_id'].append(outfit_node_ids)
            feedback_data['predicted_score'].append(score)
            feedback_data['user_feedback'].append(feedback)
            
            # Check if prediction was correct
            actual_label = 1 if row['label'] == 1 else 0
            predicted_label = 1 if (feedback == 'like') else 0
            correct = actual_label == predicted_label
            feedback_data['correct'].append(correct)
            
            feedback_text = "👍 LIKE" if feedback == 'like' else "👎 DISLIKE"
            correct_emoji = "✅" if correct else "❌"
            
            print(f"{correct_emoji} Your feedback: {feedback_text}")
            print(f"   Score: {score:.4f} | Actual: {'Compatible' if actual_label else 'Incompatible'}")
            print(f"   Samples collected: {len(feedback_data['outfit_id'])}\n")
            
            # Show next outfit
            import time
            time.sleep(1)
            next_outfit()
        
        # Create feedback buttons
        like_button = widgets.Button(
            description="👍 Like",
            button_style='success',
            tooltip='This outfit looks good together',
            layout=widgets.Layout(width='120px', height='50px')
        )
        
        dislike_button = widgets.Button(
            description="👎 Dislike",
            button_style='danger',
            tooltip='This outfit does not match well',
            layout=widgets.Layout(width='120px', height='50px')
        )
        
        skip_button = widgets.Button(
            description="⏭️  Skip",
            button_style='warning',
            tooltip='Skip this outfit',
            layout=widgets.Layout(width='120px', height='50px')
        )
        
        quit_button = widgets.Button(
            description="❌ Quit",
            button_style='danger',
            tooltip='Exit feedback loop',
            layout=widgets.Layout(width='120px', height='50px')
        )
        
        like_button.on_click(lambda b: handle_feedback('like'))
        dislike_button.on_click(lambda b: handle_feedback('dislike'))
        skip_button.on_click(lambda b: next_outfit())
        quit_button.on_click(lambda b: print(f"\n✅ Feedback session ended. Collected {len(feedback_data['outfit_id'])} samples."))
        
        # Display button group
        button_box = widgets.HBox([like_button, dislike_button, skip_button, quit_button])
        display(button_box)
    
    # Start the feedback loop
    next_outfit()
    
    return feedback_data


# Initialize interactive feedback loop
print("="*70)
print("INTERACTIVE OUTFIT FEEDBACK SYSTEM")
print("="*70)
print("\nProvide feedback on outfit compatibility predictions")
print("The model will learn which outfits you like or dislike\n")

# Start the interactive session
feedback_log = interactive_feedback_loop(
    best_model,
    df_test_outfits,
    Xc_test,
    Xa_test,
    H_test,
    Dv_inv_sqrt_test,
    De_inv_test,
    device=device
)

### Preference Learning Visualization

## 17. Top-3 Models - Comprehensive Test Validation

### Evaluate the best 3 models on test dataset
This section compares the performance of the 3 best models across different metrics on the test set.


In [ ]:
import pandas as pd
import numpy as np
import torch
import json
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score, precision_score, 
    recall_score, confusion_matrix, roc_curve, auc, 
    precision_recall_curve, classification_report
)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# Top 3 models based on validation metrics
top_3_models = [
    {
        "name": "hgnn_f512_w256x256x256_h4_e256_dr10_caF_hpT_moe0_adamw_lr1e4_wd1e5",
        "rank": 1,
        "val_roc_auc": 0.9008,
        "val_f1": 0.8265,
        "val_accuracy": 0.8194
    },
    {
        "name": "hgnn_f512_w256x256x256_128_h3_e256_dr15_caF_hpT_moe0_adamw_lr1e4_wd1e5",
        "rank": 2,
        "val_roc_auc": 0.9009,
        "val_f1": 0.8146,
        "val_accuracy": 0.8191
    },
    {
        "name": "hgnn_f512_w256x256x256_128_h3_e256_dr15_caF_hpF_moe0_adamw_lr1e4_wd1e5",
        "rank": 3,
        "val_roc_auc": 0.8894,
        "val_f1": 0.8110,
        "val_accuracy": 0.8147
    }
]

print("="*80)
print("TOP 3 MODELS FOR TEST VALIDATION")
print("="*80)
for model in top_3_models:
    print(f"\n📊 Rank {model['rank']}: {model['name'][:60]}...")
    print(f"   Val ROC-AUC: {model['val_roc_auc']:.4f} | Val F1: {model['val_f1']:.4f} | Val Acc: {model['val_accuracy']:.4f}")

print("\n" + "="*80)

In [ ]:
# Load test data using load_and_process_data function
print("\n" + "="*70)
print("LOADING TEST DATA")
print("="*70)

df_test_outfits, df_test_items, img2id_test = load_and_process_data(
    TEST_OUTFITS_PATH, TEST_ITEMS_PATH, "test"
)

# Generate attribute embeddings for test items
print("\nGenerating attribute embeddings for test items...")
attribute_embeddings_test, _ = generate_attribute_embeddings(
    df_test_items, device=device
)
df_test_items["Xa"] = list(attribute_embeddings_test)
print(f"✓ Test attribute embeddings generated: shape {attribute_embeddings_test.shape}")

# Build hyperedges for test data
print("\nBuilding hyperedges for test data...")
edge_list_test = build_hyperedges_from_outfits(df_test_outfits, wardrobe_name="test", outfits_col="node_ids")
print(f"✓ Test hyperedges built: {len(edge_list_test)} hyperedges")
print(f"  Sample hyperedges (first 3): {edge_list_test[:3]}")

# Build incidence matrix and degree matrices for test
print("\nBuilding incidence matrix for test data...")
N_nodes_test = len(df_test_items)
H_test, Dv_inv_sqrt_test, De_inv_test = build_incidence_matrix(
    num_nodes=N_nodes_test,
    hyperedges=edge_list_test,
    device=device
)
print(f"✓ Test incidence matrix built:")
print(f"  Shape: {H_test.shape}")
print(f"  Dv_inv_sqrt shape: {Dv_inv_sqrt_test.shape}")
print(f"  De_inv shape: {De_inv_test.shape}")

print("\n" + "="*70)
print("✅ TEST DATA PREPARATION COMPLETE")
print("="*70)
print(f"Summary:")
print(f"  • Test items: {len(df_test_items)}")
print(f"  • Test outfits: {len(df_test_outfits)}")
print(f"  • Test hyperedges: {len(edge_list_test)}")
print(f"  • Node ID range: 0 to {N_nodes_test-1}")
print(f"  • Attribute embedding dim: {attribute_embeddings_test.shape[1]}")


In [ ]:
# Function to evaluate model on test set
def evaluate_model_on_test(model, test_loader, test_ds, device, model_name, H, Dv_inv_sqrt, De_inv):
    """
    Comprehensive evaluation of model on test dataset
    """
    from tqdm import tqdm
    
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []
    
    # Get feature matrices
    Xc = test_ds.Xc.to(device) if hasattr(test_ds.Xc, 'to') else torch.tensor(test_ds.Xc, dtype=torch.float32, device=device)
    Xa = test_ds.Xa.to(device) if hasattr(test_ds.Xa, 'to') else torch.tensor(test_ds.Xa, dtype=torch.float32, device=device)
    num_nodes = Xc.shape[0]
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating batches", leave=False):
            # Get batch data
            nodes = batch['nodes'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device).float()
            
            # Clamp node indices to valid range
            nodes_clamped = torch.clamp(nodes, 0, num_nodes - 1)
            
            # Forward pass - model returns outfit scores directly
            outfit_scores, _, _, _ = model(Xc, Xa, H, Dv_inv_sqrt, De_inv,
                                          outfit_nodes=nodes_clamped, outfit_mask=mask)
            
            # Get probabilities
            probs = outfit_scores.cpu().numpy()
            preds = (probs > 0.5).astype(int).flatten()
            
            all_probs.extend(probs.flatten())
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy().flatten())
    
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    
    # Calculate metrics
    metrics = {
        'model_name': model_name,
        'roc_auc': roc_auc_score(all_labels, all_probs),
        'f1': f1_score(all_labels, all_preds),
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds),
        'recall': recall_score(all_labels, all_preds),
        'tp': np.sum((all_preds == 1) & (all_labels == 1)),
        'fp': np.sum((all_preds == 1) & (all_labels == 0)),
        'tn': np.sum((all_preds == 0) & (all_labels == 0)),
        'fn': np.sum((all_preds == 0) & (all_labels == 1)),
    }
    
    # Calculate MSE
    mse = np.mean((all_probs - all_labels) ** 2)
    metrics['mse'] = mse
    
    # Calculate specificity and sensitivity
    metrics['sensitivity'] = metrics['tp'] / (metrics['tp'] + metrics['fn']) if (metrics['tp'] + metrics['fn']) > 0 else 0
    metrics['specificity'] = metrics['tn'] / (metrics['tn'] + metrics['fp']) if (metrics['tn'] + metrics['fp']) > 0 else 0
    
    return metrics, all_probs, all_preds, all_labels

print("✅ Evaluation function defined")

In [ ]:
# Build test data loaders
print("\n" + "="*70)
print("BUILDING TEST DATASETS AND LOADERS")
print("="*70)

# Build Xc and Xa feature matrices for test items
print("\nBuilding feature matrices for test items...")
Xc_test, Xa_test = build_Xc_Xa_from_clothes_df(
    df_test_items,
    img_col="img_embedding",
    attr_col="Xa",
    node_id_col="node_id"
)
print(f"✓ Feature matrices built:")
print(f"  Xc_test shape: {Xc_test.shape}")
print(f"  Xa_test shape: {Xa_test.shape}")

# Parse test outfit node lists
def _parse_node_list_test(x):
    """Parse node list from various formats."""
    if isinstance(x, (list, tuple, np.ndarray)):
        return [int(i) for i in x]
    try:
        return [int(i) for i in eval(x)]
    except Exception:
        s = str(x).strip("[]")
        if s == "":
            return []
        return [int(i) for i in s.split(",")]

test_node_lists = [_parse_node_list_test(x) for x in df_test_outfits['node_ids'].values]
test_labels = np.array(df_test_outfits['label'].values, dtype=np.float32)

print(f"\n✓ Test outfits parsed:")
print(f"  Total outfits: {len(test_node_lists)}")
print(f"  Labels shape: {test_labels.shape}")

# Create test dataset
print("\nCreating test dataset...")
test_ds = NodeIdPaddedOutfitDataset(
    outfits_node_ids=test_node_lists,
    labels=test_labels,
    Xc=Xc_test,
    Xa=Xa_test,
    pad_node_id=0,
    max_len=max(len(o) for o in test_node_lists),
    device=device
)

# Create test loader
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=padded_collate)

print(f"✓ Test dataset created: {len(test_ds)} samples")
print(f"✓ Test loader created: {len(test_loader)} batches")
print(f"  Batch size: 32")
print(f"  Max outfit length: {test_ds.max_len}")

# Verify test loader
print("\n✓ Verifying test loader...")
test_batch = next(iter(test_loader))
print(f"  Batch nodes shape: {test_batch['nodes'].shape}")
print(f"  Batch mask shape: {test_batch['mask'].shape}")
print(f"  Batch label shape: {test_batch['label'].shape}")
print(f"  Number of batches: {len(test_loader)}")

print("\n" + "="*70)
print("✅ TEST DATA LOADING COMPLETE")
print("="*70)


In [ ]:
# Load and evaluate all 3 models
print("\n" + "="*80)
print("LOADING AND EVALUATING TOP 3 MODELS ON TEST SET")
print("="*80)

test_results = {}
all_metrics = []

for model_info in top_3_models:
    model_name = model_info['name']
    model_path = CHECKPOINTS_PATH / model_name / 'model.pt'
    summary_path = CHECKPOINTS_PATH / model_name / 'summary.json'
    
    print(f"\n📂 Loading Model {model_info['rank']}: {model_name[:50]}...")
    
    if not model_path.exists():
        print(f"⚠️  Model not found at {model_path}")
        continue
    
    # Load summary (contains model_cfg)
    with open(summary_path) as f:
        summary_data = json.load(f)
    
    model_cfg = summary_data['model_cfg']
    
    # Create model with correct parameter names
    model = FashionHyperGraphModel(
        clip_embed_dim=model_cfg['clip_embed_dim'],
        attr_embed_dim=model_cfg['attr_embed_dim'],
        fusion_hidden=model_cfg['fusion_hidden'],
        hgnn_hidden_list=model_cfg['hgnn_hidden_list'],
        final_embedding_dim=model_cfg['final_embedding_dim'],
        dropout=model_cfg['dropout'],
        attn_heads=model_cfg['attn_heads'],
        use_hgnn=model_cfg['use_hgnn'],
        use_cross_attention=model_cfg['use_cross_attention'],
        use_hierarchical_pooling=model_cfg['use_hierarchical_pooling'],
        use_moe=model_cfg['use_moe'],
        num_experts=model_cfg['num_experts']
    ).to(device)
    
    # Load weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    print(f"✅ Model loaded successfully")
    
    # Evaluate
    print(f"⏳ Evaluating on test set...")
    metrics, probs, preds, labels = evaluate_model_on_test(
        model, test_loader, test_ds, device, model_name, 
        H_test, Dv_inv_sqrt_test, De_inv_test
    )
    
    test_results[model_name] = {
        'metrics': metrics,
        'probs': probs,
        'preds': preds,
        'labels': labels
    }
    
    all_metrics.append(metrics)
    
    # Print results
    print(f"\n🎯 TEST RESULTS:")
    print(f"   ROC-AUC:  {metrics['roc_auc']:.4f}")
    print(f"   F1 Score: {metrics['f1']:.4f}")
    print(f"   Accuracy: {metrics['accuracy']:.4f}")
    print(f"   Precision: {metrics['precision']:.4f}")
    print(f"   Recall: {metrics['recall']:.4f}")
    print(f"   Sensitivity: {metrics['sensitivity']:.4f}")
    print(f"   Specificity: {metrics['specificity']:.4f}")
    print(f"   MSE: {metrics['mse']:.4f}")
    print(f"   TP: {metrics['tp']} | FP: {metrics['fp']} | TN: {metrics['tn']} | FN: {metrics['fn']}")

print("\n" + "="*80)

In [ ]:
# Create comparison dataframe
comparison_df = pd.DataFrame(all_metrics)
comparison_df = comparison_df[['model_name', 'roc_auc', 'f1', 'accuracy', 'precision', 'recall', 
                                'sensitivity', 'specificity', 'mse', 'tp', 'fp', 'tn', 'fn']]

# Shorten model names for display
comparison_df['model_short'] = comparison_df['model_name'].str.replace(
    'hgnn_f512_w', 'w_').str.replace('_adamw_lr1e4_wd1e5', '')
comparison_df['model_short'] = comparison_df['model_short'].str[:40] + '...'

print("\n📊 COMPARISON TABLE - All Metrics")
print(comparison_df.to_string(index=False))

# Summary statistics
print("\n" + "="*80)
print("TEST vs VALIDATION COMPARISON")
print("="*80)

comparison_summary = pd.DataFrame()
for i, model_info in enumerate(top_3_models):
    model_name = model_info['name']
    test_metrics = test_results[model_name]['metrics']
    
    summary_row = {
        'Rank': model_info['rank'],
        'Model': model_name[:45] + '...',
        'Val ROC-AUC': model_info['val_roc_auc'],
        'Test ROC-AUC': test_metrics['roc_auc'],
        'ROC-AUC Gap': round(model_info['val_roc_auc'] - test_metrics['roc_auc'], 4),
        'Val F1': model_info['val_f1'],
        'Test F1': test_metrics['f1'],
        'F1 Gap': round(model_info['val_f1'] - test_metrics['f1'], 4),
    }
    comparison_summary = pd.concat([comparison_summary, pd.DataFrame([summary_row])], ignore_index=True)

print(comparison_summary.to_string(index=False))

print("\n💡 Interpretation:")
print("   - Negative gap = Better test performance (overfitting avoided)")
print("   - Positive gap = Validation was optimistic (minor overfitting)")
print("   - Small gap (<0.03) = Good generalization")

print("\n" + "="*80)

In [ ]:
# Visualization: ROC Curves Comparison with Plotly
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'Model {idx+1}' for idx in range(3)],
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}]]
)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for idx, (model_name, results) in enumerate(test_results.items()):
    labels = results['labels']
    probs = results['probs']
    
    # Calculate ROC curve
    fpr, tpr, _ = roc_curve(labels, probs)
    roc_auc = auc(fpr, tpr)
    
    # Add ROC curve
    fig.add_trace(
        go.Scatter(
            x=fpr, y=tpr,
            mode='lines',
            name=f'ROC-AUC = {roc_auc:.4f}',
            line=dict(color=colors[idx], width=2.5),
            fill='tozeroy',
            fillcolor=colors[idx],
            opacity=0.3,
            showlegend=(idx == 0)
        ),
        row=1, col=idx+1
    )
    
    # Add random classifier line
    fig.add_trace(
        go.Scatter(
            x=[0, 1], y=[0, 1],
            mode='lines',
            name='Random',
            line=dict(color='black', width=1.5, dash='dash'),
            showlegend=(idx == 0),
            hoverinfo='skip'
        ),
        row=1, col=idx+1
    )

fig.update_xaxes(title_text="False Positive Rate", row=1, col=2)
fig.update_yaxes(title_text="True Positive Rate", row=1, col=1)

fig.update_layout(
    title_text="ROC Curves - Top 3 Models on Test Set",
    height=500,
    width=1400,
    hovermode='closest'
)

fig.write_html(str(CHECKPOINTS_PATH / 'test_roc_curves_top3.html'))
fig.show()

print("✅ ROC curves saved")


In [ ]:
# Visualization: Confusion Matrices with Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'Model {idx+1}' for idx in range(3)],
    specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}, {'type': 'heatmap'}]]
)

for idx, (model_name, results) in enumerate(test_results.items()):
    labels = results['labels']
    preds = results['preds']
    
    # Calculate confusion matrix
    cm = confusion_matrix(labels, preds)
    
    # Create heatmap
    heatmap = go.Heatmap(
        z=cm,
        x=['Negative', 'Positive'],
        y=['Negative', 'Positive'],
        colorscale='Blues',
        text=cm,
        texttemplate='%{text}',
        textfont={"size": 12},
        showscale=(idx == 2),  # Only show colorbar on last subplot
        colorbar=dict(len=0.5, y=0.5) if idx == 2 else None
    )
    
    fig.add_trace(heatmap, row=1, col=idx+1)

fig.update_layout(
    title_text="Confusion Matrices - Top 3 Models on Test Set",
    height=400,
    width=1400,
    showlegend=False
)

fig.update_yaxes(title_text="True Label", row=1, col=1)
fig.update_xaxes(title_text="Predicted Label", row=1, col=3)

fig.write_html(str(CHECKPOINTS_PATH / 'test_confusion_matrices_top3.html'))
fig.show()

print("✅ Confusion matrices saved")


In [ ]:
# Visualization: Metrics Comparison Bar Charts with Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

metrics_to_plot = ['roc_auc', 'f1', 'accuracy', 'precision', 'recall']
colors_models = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig = make_subplots(
    rows=1, cols=5,
    subplot_titles=[m.upper() for m in metrics_to_plot],
    specs=[[{'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}, {'type': 'bar'}]]
)

for col_idx, metric in enumerate(metrics_to_plot):
    values = [all_metrics[i][metric] for i in range(3)]
    
    fig.add_trace(
        go.Bar(
            x=[f'M{i+1}' for i in range(3)],
            y=values,
            marker=dict(color=colors_models),
            text=[f'{v:.4f}' for v in values],
            textposition='outside',
            showlegend=False,
            hovertemplate='<b>%{x}</b><br>%{y:.4f}<extra></extra>'
        ),
        row=1, col=col_idx+1
    )
    
    fig.update_yaxes(range=[0.75, 1.0], row=1, col=col_idx+1)
    fig.update_yaxes(title_text="Score" if col_idx == 0 else "", row=1, col=col_idx+1)

fig.update_layout(
    title_text="Detailed Metrics Comparison - Top 3 Models on Test Set",
    height=500,
    width=1600,
    showlegend=False
)

fig.write_html(str(CHECKPOINTS_PATH / 'test_metrics_comparison_top3.html'))
fig.show()

print("✅ Metrics comparison chart saved")


In [ ]:
# Precision-Recall Curves Comparison with Plotly
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'Model {idx+1}' for idx in range(3)],
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}]]
)

colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for idx, (model_name, results) in enumerate(test_results.items()):
    labels = results['labels']
    probs = results['probs']
    
    # Calculate PR curve
    precision, recall, _ = precision_recall_curve(labels, probs)
    pr_auc = auc(recall, precision)
    
    # Add PR curve
    fig.add_trace(
        go.Scatter(
            x=recall, y=precision,
            mode='lines',
            name=f'PR-AUC = {pr_auc:.4f}',
            line=dict(color=colors[idx], width=2.5),
            fill='tozeroy',
            fillcolor=colors[idx],
            opacity=0.3,
            showlegend=(idx == 0)
        ),
        row=1, col=idx+1
    )
    
    # Add baseline
    baseline = np.sum(labels==1) / len(labels)
    fig.add_hline(y=baseline, line_dash="dash", line_color="black", 
                  annotation_text="Random", row=1, col=idx+1)

fig.update_xaxes(title_text="Recall", row=1, col=2)
fig.update_yaxes(title_text="Precision", row=1, col=1)

fig.update_layout(
    title_text="Precision-Recall Curves - Top 3 Models on Test Set",
    height=500,
    width=1400,
    hovermode='closest'
)

fig.write_html(str(CHECKPOINTS_PATH / 'test_precision_recall_curves_top3.html'))
fig.show()

print("✅ Precision-Recall curves saved")


In [ ]:
# Detailed Classification Reports
print("\n" + "="*80)
print("DETAILED CLASSIFICATION REPORTS - TEST SET")
print("="*80)

for idx, (model_name, results) in enumerate(test_results.items()):
    print(f"\n{'='*80}")
    print(f"📊 MODEL {idx+1}: {model_name}")
    print(f"{'='*80}")
    
    labels = results['labels']
    preds = results['preds']
    
    report = classification_report(labels, preds, target_names=['Negative', 'Positive'], 
                                   digits=4, output_dict=False)
    print(report)

print("="*80)

In [ ]:
# Save detailed test results to CSV
test_results_csv = pd.DataFrame()

for idx, (model_name, results) in enumerate(test_results.items()):
    test_results_csv = pd.concat([test_results_csv, pd.DataFrame({
        'model_rank': [top_3_models[idx]['rank']] * len(results['labels']),
        'model_name': [model_name] * len(results['labels']),
        'true_label': results['labels'],
        'predicted_label': results['preds'],
        'predicted_probability': results['probs'],
        'prediction_correct': (results['preds'] == results['labels']).astype(int)
    })], ignore_index=True)

# Save to CSV
csv_path = CHECKPOINTS_PATH / 'test_predictions_top3_models.csv'
test_results_csv.to_csv(csv_path, index=False)
print(f"\n✅ Test predictions saved to {csv_path}")
print(f"   Total predictions: {len(test_results_csv)}")
print(f"   Columns: {list(test_results_csv.columns)}")

In [ ]:
# Summary Report JSON
test_summary = {
    'timestamp': datetime.now().isoformat(),
    'test_set_info': {
        'total_samples': len(test_ds),
        'num_batches': len(test_loader),
        'positive_samples': int(np.sum([result['labels'] for result in test_results.values()][0])),
        'negative_samples': int(np.sum(1 - np.array([result['labels'] for result in test_results.values()][0]))),
    },
    'model_rankings': []
}

# Add detailed results for each model
for idx, (model_name, results) in enumerate(test_results.items()):
    metrics = all_metrics[idx]
    
    ranking_entry = {
        'rank': idx + 1,
        'model_name': model_name,
        'validation_metrics': {
            'roc_auc': top_3_models[idx]['val_roc_auc'],
            'f1': top_3_models[idx]['val_f1'],
            'accuracy': top_3_models[idx]['val_accuracy']
        },
        'test_metrics': {
            'roc_auc': float(metrics['roc_auc']),
            'f1': float(metrics['f1']),
            'accuracy': float(metrics['accuracy']),
            'precision': float(metrics['precision']),
            'recall': float(metrics['recall']),
            'sensitivity': float(metrics['sensitivity']),
            'specificity': float(metrics['specificity']),
            'mse': float(metrics['mse']),
        },
        'confusion_matrix': {
            'tp': int(metrics['tp']),
            'fp': int(metrics['fp']),
            'tn': int(metrics['tn']),
            'fn': int(metrics['fn']),
        },
        'generalization_gap': {
            'roc_auc_gap': float(top_3_models[idx]['val_roc_auc'] - metrics['roc_auc']),
            'f1_gap': float(top_3_models[idx]['val_f1'] - metrics['f1']),
            'accuracy_gap': float(top_3_models[idx]['val_accuracy'] - metrics['accuracy']),
        }
    }
    
    test_summary['model_rankings'].append(ranking_entry)

# Save summary
summary_path = CHECKPOINTS_PATH / 'test_validation_summary_top3.json'
with open(summary_path, 'w') as f:
    json.dump(test_summary, f, indent=2)

print(f"\n✅ Test summary saved to {summary_path}")

In [ ]:
# Final Winner Analysis
print("\n" + "="*80)
print("🏆 FINAL WINNER ANALYSIS")
print("="*80)

# Determine winner based on multiple criteria
winner_scores = []

for idx, (model_name, results) in enumerate(test_results.items()):
    metrics = all_metrics[idx]
    
    # Calculate composite score (weighted average)
    weighted_score = (
        metrics['roc_auc'] * 0.35 +      # ROC-AUC is important for ranking
        metrics['f1'] * 0.35 +             # F1 balances precision and recall
        metrics['accuracy'] * 0.15 +       # Accuracy
        (1 - metrics['mse']) * 0.15       # Inverse MSE
    )
    
    winner_scores.append({
        'rank': idx + 1,
        'model_name': model_name,
        'test_roc_auc': metrics['roc_auc'],
        'test_f1': metrics['f1'],
        'test_accuracy': metrics['accuracy'],
        'test_mse': metrics['mse'],
        'weighted_score': weighted_score
    })

winner_df = pd.DataFrame(winner_scores).sort_values('weighted_score', ascending=False).reset_index(drop=True)
winner_df['final_rank'] = range(1, len(winner_df) + 1)

print("\n📊 Models Ranked by Composite Score:")
print(winner_df[['final_rank', 'model_name', 'test_roc_auc', 'test_f1', 'test_accuracy', 'weighted_score']].to_string(index=False))

print("\n" + "="*80)
print(f"🥇 RECOMMENDED MODEL: Rank {winner_df.iloc[0]['final_rank']}")
print(f"   Model: {winner_df.iloc[0]['model_name']}")
print(f"   Test ROC-AUC: {winner_df.iloc[0]['test_roc_auc']:.4f}")
print(f"   Test F1: {winner_df.iloc[0]['test_f1']:.4f}")
print(f"   Test Accuracy: {winner_df.iloc[0]['test_accuracy']:.4f}")
print(f"   Composite Score: {winner_df.iloc[0]['weighted_score']:.4f}")
print("="*80)

In [ ]:
# Visualization: Overall Ranking with Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

model_labels = [f"M{i+1}" for i in range(3)]
winner_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['ROC-AUC Comparison', 'F1 Score Comparison', 
                    'Accuracy Comparison', 'Overall Ranking Score'],
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

# ROC-AUC
roc_values = [all_metrics[i]['roc_auc'] for i in range(3)]
fig.add_trace(
    go.Bar(x=model_labels, y=roc_values, marker=dict(color=winner_colors),
           text=[f'{v:.4f}' for v in roc_values], textposition='outside',
           showlegend=False, hovertemplate='<b>%{x}</b><br>ROC-AUC: %{y:.4f}<extra></extra>'),
    row=1, col=1
)

# F1 Score
f1_values = [all_metrics[i]['f1'] for i in range(3)]
fig.add_trace(
    go.Bar(x=model_labels, y=f1_values, marker=dict(color=winner_colors),
           text=[f'{v:.4f}' for v in f1_values], textposition='outside',
           showlegend=False, hovertemplate='<b>%{x}</b><br>F1: %{y:.4f}<extra></extra>'),
    row=1, col=2
)

# Accuracy
acc_values = [all_metrics[i]['accuracy'] for i in range(3)]
fig.add_trace(
    go.Bar(x=model_labels, y=acc_values, marker=dict(color=winner_colors),
           text=[f'{v:.4f}' for v in acc_values], textposition='outside',
           showlegend=False, hovertemplate='<b>%{x}</b><br>Accuracy: %{y:.4f}<extra></extra>'),
    row=2, col=1
)

# Composite Score
composite_values = winner_df.sort_values('final_rank')['weighted_score'].values[:3]
fig.add_trace(
    go.Bar(x=model_labels, y=composite_values, marker=dict(color=winner_colors),
           text=[f'{v:.4f}' for v in composite_values], textposition='outside',
           showlegend=False, hovertemplate='<b>%{x}</b><br>Score: %{y:.4f}<extra></extra>'),
    row=2, col=2
)

fig.update_yaxes(range=[0.88, 0.91], row=1, col=1)
fig.update_yaxes(range=[0.805, 0.830], row=1, col=2)
fig.update_yaxes(range=[0.80, 0.825], row=2, col=1)
fig.update_yaxes(range=[0.814, 0.823], row=2, col=2)

fig.update_layout(
    title_text="Top-3 Models Test Performance - Final Ranking",
    height=700,
    width=1200,
    showlegend=False
)

fig.write_html(str(CHECKPOINTS_PATH / 'test_final_ranking_top3.html'))
fig.show()

print("✅ Final ranking visualization saved")


## 12. Simple GNN Model - Graph Neural Network for Outfit Compatibility

In [27]:
# ============================================
# Simple Graph Neural Network Model
# ============================================

class GraphConvLayer(nn.Module):
    """Simple Graph Convolutional Layer."""
    def __init__(self, in_dim, out_dim, use_bias=True):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=use_bias)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, X, adj):
        """
        Forward pass with adjacency matrix.
        X: (N, in_dim) - node features (on GPU)
        adj: (N, N) - adjacency matrix (on CPU, can be sparse)
        Returns: (N, out_dim)
        
        Uses sparse matrix multiplication to avoid dense conversion overhead.
        """
        # Handle sparse adjacency matrix on CPU with features on GPU
        if adj.is_sparse:
            # Use sparse-dense multiplication: adj (sparse) @ X (dense)
            # Move X to CPU temporarily for sparse-dense multiplication
            X_cpu = X.cpu()
            adj_coalesced = adj.coalesce()
            
            # Perform sparse matrix multiplication on CPU
            out_cpu = torch.sparse.mm(adj_coalesced, X_cpu)  # (N, in_dim)
            
            # Move result back to original device
            out = out_cpu.to(X.device)
        else:
            # Dense case: move adj to X's device and multiply
            adj_device = adj.to(X.device)
            out = adj_device @ X  # (N, in_dim)
        
        out = self.linear(out)  # (N, out_dim)
        out = F.relu(out)
        out = self.norm(out)
        return out


class SimpleGNN(nn.Module):
    """Simple Graph Neural Network for outfit compatibility."""
    def __init__(self, in_dim, hidden_dim=128, out_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)
        
        # Initial projection
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        
        # Graph convolutional layers
        self.gc_layers = nn.ModuleList([
            GraphConvLayer(hidden_dim if i > 0 else hidden_dim, hidden_dim)
            for i in range(num_layers)
        ])
        
        # Output projection
        self.output_proj = nn.Linear(hidden_dim, out_dim)
        self.out_norm = nn.LayerNorm(out_dim)

    def forward(self, X, adj):
        """
        X: (N, in_dim) - node features (on GPU)
        adj: (N, N) - adjacency matrix (on CPU, can be sparse)
        Returns: (N, out_dim)
        """
        x = self.input_proj(X)
        x = F.relu(x)
        x = self.dropout(x)
        
        for gc_layer in self.gc_layers:
            x = gc_layer(x, adj)
            x = self.dropout(x)
        
        x = self.output_proj(x)
        x = F.relu(x)
        x = self.out_norm(x)
        x = F.normalize(x, p=2, dim=-1)
        return x


class GNNOutfitScorer(nn.Module):
    """GNN-based outfit compatibility scorer."""
    def __init__(self, in_dim, hidden_dim=128, gnn_dim=64, num_gnn_layers=2, 
                 attn_heads=4, dropout=0.2):
        super().__init__()
        self.gnn = SimpleGNN(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=gnn_dim,
            num_layers=num_gnn_layers,
            dropout=dropout
        )
        
        # Attention pooling
        self.attn_pool = MultiHeadAttnPool(gnn_dim, n_heads=attn_heads, dropout=dropout)
        
        # Scoring head
        self.score_head = nn.Sequential(
            nn.Linear(gnn_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, X, adj, outfit_nodes, outfit_mask):
        """
        X: (N, in_dim) - node features (on GPU)
        adj: (N, N) - adjacency matrix (on CPU, sparse)
        outfit_nodes: (B, max_len) - batch of outfit node indices
        outfit_mask: (B, max_len) - mask for padding
        Returns: outfit_scores (B,)
        """
        # Apply GNN to all nodes
        # Note: GNN handles CPU adjacency with GPU features internally
        node_embeddings = self.gnn(X, adj)  # (N, gnn_dim)
        
        # Extract outfit embeddings
        emb = node_embeddings[outfit_nodes]  # (B, max_len, gnn_dim)
        
        # Attention pooling
        pooled, _ = self.attn_pool(emb, outfit_mask)  # (B, gnn_dim)
        
        # Score prediction
        scores = self.score_head(pooled).squeeze(-1)  # (B,)
        return scores, node_embeddings


def build_adjacency_matrix(hyperedges, num_nodes, device='cpu', normalize=True):
    """
    Build adjacency matrix from hyperedges.
    Items are connected if they appear in the same outfit (hyperedge).
    
    IMPORTANT: normalize=False to avoid dense matrix conversion on GPU
    """
    edge_list = []
    
    for hyperedge in hyperedges:
        # Connect all pairs of nodes in each hyperedge
        for i in range(len(hyperedge)):
            for j in range(i + 1, len(hyperedge)):
                node_i, node_j = hyperedge[i], hyperedge[j]
                if 0 <= node_i < num_nodes and 0 <= node_j < num_nodes:
                    edge_list.append((node_i, node_j))
                    edge_list.append((node_j, node_i))  # Undirected
    
    if len(edge_list) == 0:
        # Empty graph - return identity matrix as sparse
        diag_indices = torch.arange(num_nodes, dtype=torch.long)
        diag_edges = torch.stack([diag_indices, diag_indices])
        diag_values = torch.ones(num_nodes)
        adj = torch.sparse_coo_tensor(diag_edges, diag_values, (num_nodes, num_nodes))
        return adj.to(device)
    
    # Convert to COO format ON CPU FIRST
    edges = torch.tensor(edge_list, dtype=torch.long).t()
    values = torch.ones(edges.shape[1])
    
    # Create sparse adjacency matrix
    adj = torch.sparse_coo_tensor(edges, values, (num_nodes, num_nodes))
    
    # Add self-loops
    diag_indices = torch.arange(num_nodes, dtype=torch.long)
    diag_edges = torch.stack([diag_indices, diag_indices])
    diag_values = torch.ones(num_nodes)
    diag = torch.sparse_coo_tensor(diag_edges, diag_values, (num_nodes, num_nodes))
    
    adj = adj + diag
    adj = adj.coalesce()
    
    # Move to device WHILE SPARSE
    return adj.to(device)
    
    # NOTE: We skip normalization to avoid dense matrix conversion
    # Graph convolutions will work directly on sparse tensors




In [55]:
class HypergraphConvLayer(nn.Module):
    """Simplified Hypergraph Convolutional Layer."""
    def __init__(self, in_dim, out_dim, use_bias=True):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=use_bias)
        self.norm = nn.LayerNorm(out_dim)

    def forward(self, X, H, Dv_inv_sqrt, De_inv):
        """
        Simplified hypergraph convolution using hypergraph structure.
        X: (N, in_dim) - node features (on GPU)
        H: (N, E) - hypergraph incidence matrix (sparse)
        Dv_inv_sqrt, De_inv: degree matrices (can be ignored for simplification)
        Returns: (N, out_dim)
        """
        # Simplified approach: use hypergraph aggregation directly
        # H @ H^T approximates neighbor aggregation through hyperedges
        # Result: (N, N) @ (N, in_dim) = (N, in_dim)
        
        try:
            # Handle sparse H matrix
            if H.is_sparse:
                H_cpu = H.cpu().coalesce()
                X_cpu = X.cpu()
                # H @ H^T (sparse @ sparse.t) -> (N, N)
                HHt = torch.sparse.mm(H_cpu, H_cpu.t().coalesce())
                # Sparse-dense: (N, N) @ (N, in_dim)
                out_cpu = torch.sparse.mm(HHt.coalesce(), X_cpu)
                out = out_cpu.to(X.device)
            else:
                # Dense case
                HHt = H @ H.t()  # (N, N)
                out = HHt @ X      # (N, in_dim)
        except:
            # Fallback: just use X as is (hypergraph structure not applied)
            out = X
        
        out = self.linear(out)  # (N, out_dim)
        out = F.relu(out)
        out = self.norm(out)
        return out


class SimpleHGNN(nn.Module):
    """Simple Hypergraph Neural Network for outfit compatibility."""
    def __init__(self, in_dim, hidden_dim=128, out_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim
        self.num_layers = num_layers
        self.dropout = nn.Dropout(dropout)
        
        # Initial projection
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        
        # Hypergraph convolutional layers
        self.hgc_layers = nn.ModuleList([
            HypergraphConvLayer(hidden_dim if i > 0 else hidden_dim, hidden_dim)
            for i in range(num_layers)
        ])
        
        # Output projection
        self.output_proj = nn.Linear(hidden_dim, out_dim)
        self.out_norm = nn.LayerNorm(out_dim)

    def forward(self, X, H, Dv_inv_sqrt, De_inv):
        """
        X: (N, in_dim) - node features (on GPU)
        H: (N, E) - hypergraph incidence matrix
        Dv_inv_sqrt, De_inv: degree matrices (used in convolution but simplified)
        Returns: (N, out_dim)
        """
        x = self.input_proj(X)
        x = F.relu(x)
        x = self.dropout(x)
        
        for hgc_layer in self.hgc_layers:
            x = hgc_layer(x, H, Dv_inv_sqrt, De_inv)
            x = self.dropout(x)
        
        x = self.output_proj(x)
        x = F.relu(x)
        x = self.out_norm(x)
        x = F.normalize(x, p=2, dim=-1)
        return x


class HGNNOutfitScorer(nn.Module):
    """HGNN-based outfit compatibility scorer."""
    def __init__(self, in_dim, hidden_dim=128, hgnn_dim=64, num_hgnn_layers=2, 
                 attn_heads=4, dropout=0.2):
        super().__init__()
        self.hgnn = SimpleHGNN(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            out_dim=hgnn_dim,
            num_layers=num_hgnn_layers,
            dropout=dropout
        )
        
        # Attention pooling
        self.attn_pool = MultiHeadAttnPool(hgnn_dim, n_heads=attn_heads, dropout=dropout)
        
        # Scoring head
        self.score_head = nn.Sequential(
            nn.Linear(hgnn_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, X, H, Dv_inv_sqrt, De_inv, outfit_nodes, outfit_mask):
        """
        X: (N, in_dim) - node features (on GPU)
        H: (N, E) - hypergraph incidence matrix
        Dv_inv_sqrt, De_inv: degree matrices
        outfit_nodes: (B, max_len) - batch of outfit node indices
        outfit_mask: (B, max_len) - mask for padding
        Returns: outfit_scores (B,)
        """
        # Apply HGNN to all nodes
        node_embeddings = self.hgnn(X, H, Dv_inv_sqrt, De_inv)  # (N, hgnn_dim)
        
        # Extract outfit embeddings
        emb = node_embeddings[outfit_nodes]  # (B, max_len, hgnn_dim)
        
        # Attention pooling
        pooled, _ = self.attn_pool(emb, outfit_mask)  # (B, hgnn_dim)
        
        # Score prediction
        scores = self.score_head(pooled).squeeze(-1)  # (B,)
        return scores, node_embeddings

In [28]:
def train_gnn_model(
    model, optimizer, train_loader, val_loader,
    adj, Xc, Xa,
    device="cpu", epochs=20,
    early_stop_patience=10, early_stop_min_delta=0.0
):
    """Train GNN model with early stopping. ADJ STAYS ON CPU!"""
    from tqdm import tqdm
    
    model.to(device)
    
    # Ensure features are numpy first, then convert to tensor
    if isinstance(Xc, torch.Tensor):
        Xc = Xc.cpu().numpy()
    if isinstance(Xa, torch.Tensor):
        Xa = Xa.cpu().numpy()
    
    # Convert features to torch tensors on GPU
    Xc_tensor = torch.tensor(Xc, dtype=torch.float32, device=device)
    Xa_tensor = torch.tensor(Xa, dtype=torch.float32, device=device)
    X_combined = torch.cat([Xc_tensor, Xa_tensor], dim=1)  # (N, combined_dim)
    
    # Keep adjacency matrix on CPU - sparse CUDA doesn't support all ops!
    adj_cpu = adj.to('cpu') if hasattr(adj, 'to') else adj
    
    num_nodes = Xc.shape[0]
    
    best_val_loss = float("inf")
    best_state = None
    best_metrics = {}
    num_bad_epochs = 0
    
    history = {
        "train_loss": [], "val_loss": [], "mse": [], "roc_auc": [],
        "accuracy": [], "f1": []
    }
    
    print(f"\n{'='*100}")
    print(f"Training GNN on {device} (Adjacency on CPU) | Epochs: {epochs} | Batches/epoch: {len(train_loader)}")
    print(f"{'='*100}\n")
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch:3d}/{epochs}", unit="batch", leave=False)
        for batch in pbar:
            optimizer.zero_grad()
            nodes = batch["nodes"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["label"].to(device)
            
            # Clamp node indices to valid range
            nodes_clamped = torch.clamp(nodes, 0, num_nodes - 1)
            
            # Forward pass - features on GPU, adjacency on CPU
            scores, _ = model(X_combined, adj_cpu, nodes_clamped, mask)
            
            # Loss
            loss = F.binary_cross_entropy(scores, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
            
            pbar.set_postfix({"Loss": f"{loss.item():.6f}"})
        
        avg_train_loss = total_loss / len(train_loader.dataset)
        history["train_loss"].append(avg_train_loss)
        
        # Validation
        model.eval()
        val_loss = 0.0
        probs_list = []
        labels_list = []
        
        with torch.no_grad():
            for batch in val_loader:
                nodes = batch["nodes"].to(device)
                mask = batch["mask"].to(device)
                labels = batch["label"].to(device)
                
                nodes_clamped = torch.clamp(nodes, 0, num_nodes - 1)
                scores, _ = model(X_combined, adj_cpu, nodes_clamped, mask)
                
                loss = F.binary_cross_entropy(scores, labels)
                val_loss += loss.item() * len(labels)
                
                # ENSURE conversion to CPU BEFORE numpy
                probs_list.append(scores.detach().cpu().numpy())
                labels_list.append(labels.detach().cpu().numpy())
        
        val_loss = val_loss / len(val_loader.dataset)
        history["val_loss"].append(val_loss)
        
        probs_all = np.concatenate(probs_list)
        labels_all = np.concatenate(labels_list)
        preds = (probs_all >= 0.5).astype(int)
        
        # Compute metrics
        mse = mean_squared_error(labels_all, probs_all)
        roc_auc = roc_auc_score(labels_all, probs_all) if len(np.unique(labels_all)) > 1 else float("nan")
        accuracy = accuracy_score(labels_all, preds)
        f1 = f1_score(labels_all, preds, zero_division=0)
        
        history["mse"].append(mse)
        history["roc_auc"].append(roc_auc)
        history["accuracy"].append(accuracy)
        history["f1"].append(f1)
        
        metrics = {
            "val_loss": val_loss,
            "mse": mse,
            "roc_auc": roc_auc,
            "accuracy": accuracy,
            "f1": f1
        }
        
        # Track best
        if val_loss < best_val_loss - early_stop_min_delta:
            best_val_loss = val_loss
            best_metrics = metrics.copy()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            num_bad_epochs = 0
        else:
            num_bad_epochs += 1
        
        print(f"Epoch {epoch}/{epochs} | TrainLoss={avg_train_loss:.6f} | ValLoss={val_loss:.6f} | BestVal={best_val_loss:.6f} | ROC-AUC={roc_auc:.4f} | F1={f1:.4f}")
        
        if num_bad_epochs > early_stop_patience:
            print(f"\nEarly stopping at epoch {epoch}. Best val_loss: {best_val_loss:.6f}")
            break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    return model, best_val_loss, best_metrics, history


In [53]:
# Build adjacency matrix from training hyperedges - KEEP ON CPU!
print("Building adjacency matrix from hyperedges...")
adj_train = build_adjacency_matrix(edge_list_train, N_nodes_train, device='cpu', normalize=False)
print(f"Adjacency matrix shape: {adj_train.shape}")
print(f"Adjacency matrix is sparse: {adj_train.is_sparse}")
print(f"Non-zero elements: {adj_train._nnz()}")
print(f"Memory usage: ~{adj_train._nnz() * 8 / 1024 / 1024:.2f} MB")

# Prepare node features (concatenate CLIP and attribute embeddings)
Xc_np = train_ds.Xc.cpu().numpy() if isinstance(train_ds.Xc, torch.Tensor) else train_ds.Xc
Xa_np = train_ds.Xa.cpu().numpy() if isinstance(train_ds.Xa, torch.Tensor) else train_ds.Xa
in_dim = Xc_np.shape[1] + Xa_np.shape[1]
print(f"Feature matrices: Xc {Xc_np.shape}, Xa {Xa_np.shape}, combined dim: {in_dim}")

# Initialize GNN model with REDUCED complexity
print("\nInitializing GNN model...")
gnn_scorer = GNNOutfitScorer(
    in_dim=in_dim,
    hidden_dim=128,  # Reduced from 256
    gnn_dim=64,      # Reduced from 128
    num_gnn_layers=2,
    attn_heads=2,    # Reduced from 4
    dropout=0.2
).to(device)

gnn_optimizer = torch.optim.AdamW(gnn_scorer.parameters(), lr=1e-4, weight_decay=1e-5)
print(f"GNN Model parameters: {sum(p.numel() for p in gnn_scorer.parameters()):,}")

# Train GNN - adjacency stays on CPU!
print("\nStarting GNN training...")
gnn_model, gnn_best_val_loss, gnn_best_metrics, gnn_history = train_gnn_model(
    model=gnn_scorer,
    optimizer=gnn_optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    adj=adj_train,  # This stays on CPU
    Xc=Xc_np,
    Xa=Xa_np,
    device=device,
    epochs=50,  # Reduced from 100
    early_stop_patience=10,
    early_stop_min_delta=1e-4
)

print(f"\nGNN Training complete!")
print(f"Best validation loss: {gnn_best_val_loss:.6f}")
print(f"Best metrics: {gnn_best_metrics}")

Building adjacency matrix from hyperedges...
Adjacency matrix shape: torch.Size([123787, 123787])
Adjacency matrix is sparse: True
Non-zero elements: 894661
Memory usage: ~6.83 MB
Feature matrices: Xc (123787, 512), Xa (123787, 256), combined dim: 768

Initializing GNN model...
GNN Model parameters: 165,505

Starting GNN training...

Training GNN on cuda (Adjacency on CPU) | Epochs: 50 | Batches/epoch: 122



Epoch 1/50 | TrainLoss=0.684716 | ValLoss=0.660861 | BestVal=0.660861 | ROC-AUC=0.6608 | F1=0.5818


Epoch 2/50 | TrainLoss=0.622057 | ValLoss=0.566109 | BestVal=0.566109 | ROC-AUC=0.7846 | F1=0.6832


Epoch 3/50 | TrainLoss=0.556966 | ValLoss=0.516877 | BestVal=0.516877 | ROC-AUC=0.8421 | F1=0.7086


Epoch 4/50 | TrainLoss=0.492940 | ValLoss=0.421745 | BestVal=0.421745 | ROC-AUC=0.8940 | F1=0.8020


Epoch 5/50 | TrainLoss=0.443376 | ValLoss=0.395335 | BestVal=0.395335 | ROC-AUC=0.9063 | F1=0.8247


Epoch 6/50 | TrainLoss=0.419418 | ValLoss=0.384719 | BestVal=0.384719 | ROC-AUC=0.9091 | F1=0.8323


KeyboardInterrupt: 

In [31]:
# Save GNN model checkpoint
print("Saving GNN model checkpoint...")

gnn_checkpoint_dir = Path("../experiments_fashion_score/gnn_models")
gnn_checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Create experiment subdirectory
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
gnn_exp_name = f"simple_gnn_h256_g128_l2_h4_dr20_adamw_lr1e-4_wd1e-5_{timestamp}"
gnn_exp_dir = gnn_checkpoint_dir / gnn_exp_name
gnn_exp_dir.mkdir(parents=True, exist_ok=True)

# Save model state dict
torch.save(gnn_model.state_dict(), gnn_exp_dir / "model.pt")
print(f"✓ GNN model checkpoint saved: {gnn_exp_dir / 'model.pt'}")

# Save training history
with open(gnn_exp_dir / "history.json", "w") as f:
    gnn_history_json = {k: [float(v) for v in vals] for k, vals in gnn_history.items()}
    json.dump(gnn_history_json, f, indent=2)
print(f"✓ GNN training history saved: {gnn_exp_dir / 'history.json'}")

# Save best metrics
with open(gnn_exp_dir / "metrics.json", "w") as f:
    gnn_metrics_json = {k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                       for k, v in gnn_best_metrics.items()}
    json.dump(gnn_metrics_json, f, indent=2)
print(f"✓ GNN best metrics saved: {gnn_exp_dir / 'metrics.json'}")

# Recreate X_combined for metadata (already used Xc_np and Xa_np from earlier)
X_combined_shape_1 = Xc_np.shape[1] + Xa_np.shape[1]  # Reconstruct dimension

# Save experiment summary
gnn_summary = {
    "name": gnn_exp_name,
    "timestamp": timestamp,
    "best_val_loss": float(gnn_best_val_loss),
    "best_metrics": {k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                    for k, v in gnn_best_metrics.items()},
    "model_config": {
        "in_dim": X_combined_shape_1,
        "hidden_dim": 128,
        "gnn_dim": 64,
        "num_gnn_layers": 2,
        "attn_heads": 2,
        "dropout": 0.2
    },
    "optimizer_config": {
        "type": "AdamW",
        "lr": 1e-4,
        "weight_decay": 1e-5,
        "betas": [0.9, 0.999]
    },
    "training_config": {
        "epochs": 50,
        "batch_size": len(train_loader.dataset),
        "early_stop_patience": 10,
        "early_stop_min_delta": 1e-4
    },
    "adjacency_matrix": {
        "shape": list(adj_train.shape),
        "is_sparse": adj_train.is_sparse,
        "nnz": int(adj_train._nnz()) if adj_train.is_sparse else None,
        "source": "outfit_hyperedges"
    }
}

with open(gnn_exp_dir / "summary.json", "w") as f:
    json.dump(gnn_summary, f, indent=2)
print(f"✓ GNN summary saved: {gnn_exp_dir / 'summary.json'}")

print(f"\n{'='*70}")
print(f"GNN Experiment saved to: {gnn_exp_dir}")
print(f"{'='*70}")

Saving GNN model checkpoint...
✓ GNN model checkpoint saved: ..\experiments_fashion_score\gnn_models\simple_gnn_h256_g128_l2_h4_dr20_adamw_lr1e-4_wd1e-5_20260120_194208\model.pt
✓ GNN training history saved: ..\experiments_fashion_score\gnn_models\simple_gnn_h256_g128_l2_h4_dr20_adamw_lr1e-4_wd1e-5_20260120_194208\history.json
✓ GNN best metrics saved: ..\experiments_fashion_score\gnn_models\simple_gnn_h256_g128_l2_h4_dr20_adamw_lr1e-4_wd1e-5_20260120_194208\metrics.json
✓ GNN summary saved: ..\experiments_fashion_score\gnn_models\simple_gnn_h256_g128_l2_h4_dr20_adamw_lr1e-4_wd1e-5_20260120_194208\summary.json

GNN Experiment saved to: ..\experiments_fashion_score\gnn_models\simple_gnn_h256_g128_l2_h4_dr20_adamw_lr1e-4_wd1e-5_20260120_194208


### GNN Model Evaluation and Visualization

In [33]:
# Evaluate GNN on test set
def evaluate_gnn(model, val_loader, adj, Xc, Xa, device='cpu', threshold=0.5):
    """Evaluate GNN model."""
    model.eval()
    probs_list = []
    labels_list = []
    
    # Ensure features are numpy first, then convert to tensor
    if isinstance(Xc, torch.Tensor):
        Xc = Xc.cpu().numpy()
    if isinstance(Xa, torch.Tensor):
        Xa = Xa.cpu().numpy()
    
    # Convert features to torch tensors on device
    Xc_tensor = torch.tensor(Xc, dtype=torch.float32, device=device)
    Xa_tensor = torch.tensor(Xa, dtype=torch.float32, device=device)
    X_combined = torch.cat([Xc_tensor, Xa_tensor], dim=1)  # (N, combined_dim)
    
    # Keep adjacency matrix on CPU
    adj_cpu = adj.to('cpu') if hasattr(adj, 'to') else adj
    num_nodes = Xc.shape[0]
    
    with torch.no_grad():
        for batch in val_loader:
            nodes = batch["nodes"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["label"].to(device)
            
            # Clamp node indices to valid range
            nodes_clamped = torch.clamp(nodes, 0, num_nodes - 1)
            
            # Forward pass - features on GPU, adjacency on CPU
            scores, _ = model(X_combined, adj_cpu, nodes_clamped, mask)
            probs_list.append(scores.detach().cpu().numpy())
            labels_list.append(labels.detach().cpu().numpy())
    
    probs_all = np.concatenate(probs_list)
    labels_all = np.concatenate(labels_list)
    preds = (probs_all >= threshold).astype(int)
    
    metrics = {
        "roc_auc": roc_auc_score(labels_all, probs_all) if len(np.unique(labels_all)) > 1 else float("nan"),
        "f1": f1_score(labels_all, preds, zero_division=0),
        "accuracy": accuracy_score(labels_all, preds),
        "precision": precision_score(labels_all, preds, zero_division=0),
        "recall": recall_score(labels_all, preds, zero_division=0),
        "mse": mean_squared_error(labels_all, probs_all),
    }
    
    return metrics, probs_all, labels_all

print("Evaluating GNN on validation set...")
gnn_val_metrics, gnn_probs, gnn_labels = evaluate_gnn(
    gnn_model, val_loader, adj_train, Xc_np, Xa_np, device=device
)

print("\nGNN Validation Metrics:")
for metric, value in gnn_val_metrics.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value}")

Evaluating GNN on validation set...

GNN Validation Metrics:
  roc_auc: 0.9189
  f1: 0.8437
  accuracy: 0.8447
  precision: 0.8636
  recall: 0.8247
  mse: 0.1148


In [34]:
# Visualize GNN training curves
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Training & Validation Loss', 'ROC-AUC Score', 'F1 Score', 'Accuracy'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'scatter'}]]
)

# Training Loss
epochs_range = range(1, len(gnn_history['train_loss']) + 1)
fig.add_trace(
    go.Scatter(x=list(epochs_range), y=gnn_history['train_loss'], 
               mode='lines', name='Train Loss',
               line=dict(color='blue')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=list(epochs_range), y=gnn_history['val_loss'],
               mode='lines', name='Val Loss',
               line=dict(color='red')),
    row=1, col=1
)

# ROC-AUC
fig.add_trace(
    go.Scatter(x=list(epochs_range), y=gnn_history['roc_auc'],
               mode='lines', name='ROC-AUC',
               line=dict(color='green')),
    row=1, col=2
)

# F1 Score
fig.add_trace(
    go.Scatter(x=list(epochs_range), y=gnn_history['f1'],
               mode='lines', name='F1',
               line=dict(color='purple')),
    row=2, col=1
)

# Accuracy
fig.add_trace(
    go.Scatter(x=list(epochs_range), y=gnn_history['accuracy'],
               mode='lines', name='Accuracy',
               line=dict(color='orange')),
    row=2, col=2
)

fig.update_xaxes(title_text="Epoch", row=1, col=1)
fig.update_xaxes(title_text="Epoch", row=1, col=2)
fig.update_xaxes(title_text="Epoch", row=2, col=1)
fig.update_xaxes(title_text="Epoch", row=2, col=2)

fig.update_yaxes(title_text="Loss", row=1, col=1)
fig.update_yaxes(title_text="ROC-AUC", row=1, col=2)
fig.update_yaxes(title_text="F1 Score", row=2, col=1)
fig.update_yaxes(title_text="Accuracy", row=2, col=2)

fig.update_layout(
    title_text="Simple GNN Training Metrics",
    height=700,
    width=1200,
    hovermode='x unified'
)

fig.write_html(str(gnn_exp_dir / 'gnn_training_curves.html'))
fig.show()

print("✅ GNN training curves visualization saved")


✅ GNN training curves visualization saved


In [56]:
# ============================================
# Simple Hypergraph Neural Network Model Training
# ============================================

print("\n" + "="*100)
print("HGNN (Hypergraph Neural Network) Training")
print("="*100 + "\n")

# Initialize simplified HGNN model (similar to GNN but with hypergraph structure)
print("Initializing simplified HGNN model...")
hgnn_model = HGNNOutfitScorer(
    in_dim=768,  #
    hidden_dim=128,
    hgnn_dim=64,
    num_hgnn_layers=2,
    attn_heads=2,
    dropout=0.2
).to(device)

hgnn_optimizer = torch.optim.AdamW(hgnn_model.parameters(), lr=1e-3, weight_decay=1e-5)
print(f"HGNN Model parameters: {sum(p.numel() for p in hgnn_model.parameters()):,}")

# HGNN training function
def train_hgnn_model(
    model, optimizer, train_loader, val_loader,
    H, Dv_inv_sqrt, De_inv, Xc, Xa,
    device="cuda", epochs=100,
    early_stop_patience=15, early_stop_min_delta=1e-5
):
    """Train HGNN model with early stopping and hypergraph operators."""
    from tqdm import tqdm
    
    model.to(device)
    
    # Ensure features are numpy first, then convert to tensors on device
    if isinstance(Xc, torch.Tensor):
        Xc = Xc.cpu().numpy()
    if isinstance(Xa, torch.Tensor):
        Xa = Xa.cpu().numpy()
    
    # Combine features and convert to tensor on device
    X_combined = np.concatenate([Xc, Xa], axis=1)  # (N, 768)
    X_tensor = torch.tensor(X_combined, dtype=torch.float32, device=device)
    
    # Move hypergraph operators to device
    H = H.to(device)
    Dv_inv_sqrt = Dv_inv_sqrt.to(device)
    De_inv = De_inv.to(device)
    
    best_val_loss = float("inf")
    best_state = None
    best_metrics = {}
    num_bad_epochs = 0
    
    history = {
        "train_loss": [], "val_loss": [], "mse": [], "roc_auc": [],
        "accuracy": [], "f1": []
    }
    
    print(f"\n{'='*100}")
    print(f"Training HGNN on {device} | Epochs: {epochs} | Batches/epoch: {len(train_loader)}")
    print(f"{'='*100}\n")
    
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch:3d}/{epochs}", unit="batch", leave=False)
        for batch in pbar:
            optimizer.zero_grad()
            nodes = batch["nodes"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["label"].to(device)
            
            # Clamp node indices to valid range
            nodes_clamped = torch.clamp(nodes, 0, X_tensor.shape[0] - 1)
            
            # Forward pass with hypergraph
            scores, _ = model(X_tensor, H, Dv_inv_sqrt, De_inv, nodes_clamped, mask)
            
            # Loss
            loss = F.binary_cross_entropy(scores, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(labels)
            
            pbar.set_postfix({"Loss": f"{loss.item():.6f}"})
        
        avg_train_loss = total_loss / len(train_loader.dataset)
        history["train_loss"].append(avg_train_loss)
        
        # Validation
        model.eval()
        val_loss = 0.0
        probs_list = []
        labels_list = []
        
        with torch.no_grad():
            for batch in val_loader:
                nodes = batch["nodes"].to(device)
                mask = batch["mask"].to(device)
                labels = batch["label"].to(device)
                
                # Clamp node indices to valid range
                nodes_clamped = torch.clamp(nodes, 0, X_tensor.shape[0] - 1)
                
                # Forward pass
                scores, _ = model(X_tensor, H, Dv_inv_sqrt, De_inv, nodes_clamped, mask)
                
                loss = F.binary_cross_entropy(scores, labels)
                val_loss += loss.item() * len(labels)
                
                # Ensure conversion to CPU BEFORE numpy
                probs_list.append(scores.detach().cpu().numpy())
                labels_list.append(labels.detach().cpu().numpy())
        
        val_loss = val_loss / len(val_loader.dataset)
        history["val_loss"].append(val_loss)
        
        probs_all = np.concatenate(probs_list)
        labels_all = np.concatenate(labels_list)
        preds = (probs_all >= 0.5).astype(int)
        
        # Compute metrics
        mse = mean_squared_error(labels_all, probs_all)
        roc_auc = roc_auc_score(labels_all, probs_all) if len(np.unique(labels_all)) > 1 else float("nan")
        accuracy = accuracy_score(labels_all, preds)
        f1 = f1_score(labels_all, preds, zero_division=0)
        
        history["mse"].append(mse)
        history["roc_auc"].append(roc_auc)
        history["accuracy"].append(accuracy)
        history["f1"].append(f1)
        
        metrics = {
            "val_loss": val_loss,
            "mse": mse,
            "roc_auc": roc_auc,
            "accuracy": accuracy,
            "f1": f1
        }
        
        # Track best
        if val_loss < best_val_loss - early_stop_min_delta:
            best_val_loss = val_loss
            best_metrics = metrics.copy()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            num_bad_epochs = 0
        else:
            num_bad_epochs += 1
        
        print(f"Epoch {epoch}/{epochs} | TrainLoss={avg_train_loss:.6f} | ValLoss={val_loss:.6f} | BestVal={best_val_loss:.6f} | ROC-AUC={roc_auc:.4f} | F1={f1:.4f}")
        
        if num_bad_epochs > early_stop_patience:
            print(f"\nEarly stopping at epoch {epoch}. Best val_loss: {best_val_loss:.6f}")
            break
    
    if best_state is not None:
        model.load_state_dict(best_state)
    
    return model, best_val_loss, best_metrics, history

# Train HGNN
print("\nStarting HGNN training...")
hgnn_model, hgnn_best_val_loss, hgnn_best_metrics, hgnn_history = train_hgnn_model(
    model=hgnn_model,
    optimizer=hgnn_optimizer,
    train_loader=train_loader,
    val_loader=val_loader,
    H=H_train,
    Dv_inv_sqrt=Dv_inv_sqrt_train,
    De_inv=De_inv_train,
    Xc=Xc_np,
    Xa=Xa_np,
    device=device,
    epochs=50,
    early_stop_patience=10,
    early_stop_min_delta=1e-5
)

print(f"\nHGNN Training complete!")
print(f"Best validation loss: {hgnn_best_val_loss:.6f}")
print(f"Best metrics: {hgnn_best_metrics}")


HGNN (Hypergraph Neural Network) Training

Initializing simplified HGNN model...
HGNN Model parameters: 165,505

Starting HGNN training...

Training HGNN on cuda | Epochs: 50 | Batches/epoch: 122



Epoch   1/50:   0%|          | 0/122 [00:00<?, ?batch/s]C:\Users\gorka\AppData\Local\Temp\ipykernel_1888\3416683539.py:26: UserWarning:

Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:55.)



Epoch 1/50 | TrainLoss=0.689049 | ValLoss=0.689044 | BestVal=0.689044 | ROC-AUC=0.6266 | F1=0.1393


Epoch 2/50 | TrainLoss=0.641238 | ValLoss=0.632510 | BestVal=0.632510 | ROC-AUC=0.6301 | F1=0.4925


Epoch 3/50 | TrainLoss=0.606093 | ValLoss=0.519482 | BestVal=0.519482 | ROC-AUC=0.8382 | F1=0.6687


...

Epoch 17/50 | TrainLoss=0.224072 | ValLoss=0.363339 | BestVal=0.316813 | ROC-AUC=0.9393 | F1=0.8627


Epoch 18/50 | TrainLoss=0.219102 | ValLoss=0.335919 | BestVal=0.316813 | ROC-AUC=0.9387 | F1=0.8666


Epoch 19/50 | TrainLoss=0.207111 | ValLoss=0.364395 | BestVal=0.316813 | ROC-AUC=0.9390 | F1=0.8698


Epoch 20/50 | TrainLoss=0.204558 | ValLoss=0.329648 | BestVal=0.316813 | ROC-AUC=0.9402 | F1=0.8708


Epoch 21/50 | TrainLoss=0.200451 | ValLoss=0.381605 | BestVal=0.316813 | ROC-AUC=0.9355 | F1=0.8589

Early stopping at epoch 21. Best val_loss: 0.316813

HGNN Training complete!
Best validation loss: 0.316813
Best metrics: {'val_loss': 0.3168129639244669, 'mse': 0.09702615439891815, 'roc_auc': 0.9403604447879955, 'accuracy': 0.8693733451015004, 'f1': 0.871824480369515}


In [57]:
# Save HGNN model checkpoint
print("Saving HGNN model checkpoint...")

hgnn_checkpoint_dir = Path("../experiments_fashion_score/hgnn_models")
hgnn_checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Create experiment subdirectory with SIMPLIFIED configuration
hgnn_exp_name = f"simple_hgnn_h128_g64_l2_h2_dr20_adamw_lr1e-3_wd1e-5_{timestamp}"
hgnn_exp_dir = hgnn_checkpoint_dir / hgnn_exp_name
hgnn_exp_dir.mkdir(parents=True, exist_ok=True)

# Save model state dict
torch.save(hgnn_model.state_dict(), hgnn_exp_dir / "model.pt")
print(f"✓ HGNN model checkpoint saved: {hgnn_exp_dir / 'model.pt'}")

# Save training history
with open(hgnn_exp_dir / "history.json", "w") as f:
    hgnn_history_json = {k: [float(v) for v in vals] for k, vals in hgnn_history.items()}
    json.dump(hgnn_history_json, f, indent=2)
print(f"✓ HGNN training history saved: {hgnn_exp_dir / 'history.json'}")

# Save best metrics
with open(hgnn_exp_dir / "metrics.json", "w") as f:
    hgnn_metrics_json = {k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                        for k, v in hgnn_best_metrics.items()}
    json.dump(hgnn_metrics_json, f, indent=2)
print(f"✓ HGNN best metrics saved: {hgnn_exp_dir / 'metrics.json'}")

# Save experiment summary with SIMPLIFIED HGNN config
hgnn_summary = {
    "name": hgnn_exp_name,
    "timestamp": timestamp,
    "best_val_loss": float(hgnn_best_val_loss),
    "best_metrics": {k: (float(v) if isinstance(v, (np.floating, float)) else v) 
                    for k, v in hgnn_best_metrics.items()},
    "model_config": {
        "type": "SimpleHGNN",
        "in_dim": 768,
        "hidden_dim": 128,
        "hgnn_dim": 64,
        "num_hgnn_layers": 2,
        "attn_heads": 2,
        "dropout": 0.2,
        "use_hgnn": True,
        "note": "Simplified HGNN matching GNN complexity for fair comparison"
    },
    "optimizer_config": {
        "type": "AdamW",
        "lr": 1e-3,
        "weight_decay": 1e-5,
        "betas": [0.9, 0.999]
    },
    "training_config": {
        "epochs": 50,
        "batch_size": len(train_loader.dataset),
        "early_stop_patience": 10,
        "early_stop_min_delta": 1e-5
    },
    "hypergraph_matrix": {
        "H_shape": list(H_train.shape),
        "Dv_inv_sqrt_shape": list(Dv_inv_sqrt_train.shape),
        "De_inv_shape": list(De_inv_train.shape),
        "source": "train_outfit_hyperedges"
    }
}

with open(hgnn_exp_dir / "summary.json", "w") as f:
    json.dump(hgnn_summary, f, indent=2)
print(f"✓ HGNN summary saved: {hgnn_exp_dir / 'summary.json'}")

print(f"\n{'='*70}")
print(f"HGNN Experiment saved to: {hgnn_exp_dir}")
print(f"Architecture: Simplified HGNN with hypergraph convolutions")
print(f"{'='*70}")

Saving HGNN model checkpoint...
✓ HGNN model checkpoint saved: ..\experiments_fashion_score\hgnn_models\simple_hgnn_h128_g64_l2_h2_dr20_adamw_lr1e-3_wd1e-5_20260120_194208\model.pt
✓ HGNN training history saved: ..\experiments_fashion_score\hgnn_models\simple_hgnn_h128_g64_l2_h2_dr20_adamw_lr1e-3_wd1e-5_20260120_194208\history.json
✓ HGNN best metrics saved: ..\experiments_fashion_score\hgnn_models\simple_hgnn_h128_g64_l2_h2_dr20_adamw_lr1e-3_wd1e-5_20260120_194208\metrics.json
✓ HGNN summary saved: ..\experiments_fashion_score\hgnn_models\simple_hgnn_h128_g64_l2_h2_dr20_adamw_lr1e-3_wd1e-5_20260120_194208\summary.json

HGNN Experiment saved to: ..\experiments_fashion_score\hgnn_models\simple_hgnn_h128_g64_l2_h2_dr20_adamw_lr1e-3_wd1e-5_20260120_194208
Architecture: Simplified HGNN with hypergraph convolutions


In [58]:
# Evaluate HGNN on validation set
def evaluate_hgnn(model, val_loader, H, Dv_inv_sqrt, De_inv, Xc, Xa, device='cuda', threshold=0.5):
    """Evaluate HGNN model."""
    model.eval()
    probs_list = []
    labels_list = []
    
    # Ensure features are numpy first, then convert to tensor on device
    if isinstance(Xc, torch.Tensor):
        Xc = Xc.cpu().numpy()
    if isinstance(Xa, torch.Tensor):
        Xa = Xa.cpu().numpy()
    
    # Combine features and convert to tensor on device
    X_combined = np.concatenate([Xc, Xa], axis=1)  # (N, 768)
    X_tensor = torch.tensor(X_combined, dtype=torch.float32, device=device)
    
    # Move hypergraph operators to device
    H = H.to(device)
    Dv_inv_sqrt = Dv_inv_sqrt.to(device)
    De_inv = De_inv.to(device)
    
    with torch.no_grad():
        for batch in val_loader:
            nodes = batch["nodes"].to(device)
            mask = batch["mask"].to(device)
            labels = batch["label"].to(device)
            
            # Clamp node indices to valid range
            nodes_clamped = torch.clamp(nodes, 0, X_tensor.shape[0] - 1)
            
            # Forward pass with hypergraph
            scores, _ = model(X_tensor, H, Dv_inv_sqrt, De_inv, nodes_clamped, mask)
            
            probs_list.append(scores.detach().cpu().numpy())
            labels_list.append(labels.detach().cpu().numpy())
    
    probs_all = np.concatenate(probs_list)
    labels_all = np.concatenate(labels_list)
    preds = (probs_all >= threshold).astype(int)
    
    metrics = {
        "roc_auc": roc_auc_score(labels_all, probs_all) if len(np.unique(labels_all)) > 1 else float("nan"),
        "f1": f1_score(labels_all, preds, zero_division=0),
        "accuracy": accuracy_score(labels_all, preds),
        "precision": precision_score(labels_all, preds, zero_division=0),
        "recall": recall_score(labels_all, preds, zero_division=0),
        "mse": mean_squared_error(labels_all, probs_all),
    }
    
    return metrics, probs_all, labels_all

print("Evaluating HGNN on validation set...")
hgnn_val_metrics, hgnn_probs, hgnn_labels = evaluate_hgnn(
    hgnn_model, val_loader, H_train, Dv_inv_sqrt_train, De_inv_train, 
    Xc_np, Xa_np, device=device
)

print("\nHGNN Validation Metrics:")
for metric, value in hgnn_val_metrics.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.4f}")
    else:
        print(f"  {metric}: {value}")

Evaluating HGNN on validation set...

HGNN Validation Metrics:
  roc_auc: 0.9404
  f1: 0.8718
  accuracy: 0.8694
  precision: 0.8698
  recall: 0.8738
  mse: 0.0970


In [59]:
# ============================================
# GNN vs HGNN Comparison
# ============================================

print("\n" + "="*100)
print("Model Comparison: GNN vs HGNN")
print("="*100 + "\n")

# Create comparison table
comparison_data = {
    "Metric": ["ROC-AUC", "F1 Score", "Accuracy", "Precision", "Recall", "MSE", "Val Loss"],
    "GNN": [
        f"{gnn_val_metrics['roc_auc']:.4f}",
        f"{gnn_val_metrics['f1']:.4f}",
        f"{gnn_val_metrics['accuracy']:.4f}",
        f"{gnn_val_metrics['precision']:.4f}",
        f"{gnn_val_metrics['recall']:.4f}",
        f"{gnn_val_metrics['mse']:.4f}",
        f"{gnn_best_val_loss:.6f}"
    ],
    "HGNN": [
        f"{hgnn_val_metrics['roc_auc']:.4f}",
        f"{hgnn_val_metrics['f1']:.4f}",
        f"{hgnn_val_metrics['accuracy']:.4f}",
        f"{hgnn_val_metrics['precision']:.4f}",
        f"{hgnn_val_metrics['recall']:.4f}",
        f"{hgnn_val_metrics['mse']:.4f}",
        f"{hgnn_best_val_loss:.6f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print()

# Calculate differences
print("Difference (HGNN - GNN):")
for metric in ["roc_auc", "f1", "accuracy", "precision", "recall", "mse"]:
    diff = hgnn_val_metrics[metric] - gnn_val_metrics[metric]
    print(f"  {metric}: {diff:+.4f}")

loss_diff = hgnn_best_val_loss - gnn_best_val_loss
print(f"  val_loss: {loss_diff:+.6f}")

# Training curves comparison
fig_comparison = go.Figure()

# GNN curves
fig_comparison.add_trace(go.Scatter(
    y=gnn_history["train_loss"],
    name="GNN Training Loss",
    mode="lines",
    line=dict(color="blue", dash="solid")
))

fig_comparison.add_trace(go.Scatter(
    y=gnn_history["val_loss"],
    name="GNN Validation Loss",
    mode="lines",
    line=dict(color="blue", dash="dash")
))

# HGNN curves
fig_comparison.add_trace(go.Scatter(
    y=hgnn_history["train_loss"],
    name="HGNN Training Loss",
    mode="lines",
    line=dict(color="red", dash="solid")
))

fig_comparison.add_trace(go.Scatter(
    y=hgnn_history["val_loss"],
    name="HGNN Validation Loss",
    mode="lines",
    line=dict(color="red", dash="dash")
))

fig_comparison.update_layout(
    title="Training Curves: GNN vs HGNN",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    hovermode="x unified",
    height=500,
    template="plotly_white"
)

fig_comparison.show()

# ROC-AUC comparison
fig_metrics = go.Figure()

fig_metrics.add_trace(go.Bar(
    name="GNN",
    x=["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"],
    y=[
        gnn_val_metrics["roc_auc"],
        gnn_val_metrics["f1"],
        gnn_val_metrics["accuracy"],
        gnn_val_metrics["precision"],
        gnn_val_metrics["recall"]
    ],
    marker=dict(color="blue", opacity=0.7)
))

fig_metrics.add_trace(go.Bar(
    name="HGNN",
    x=["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"],
    y=[
        hgnn_val_metrics["roc_auc"],
        hgnn_val_metrics["f1"],
        hgnn_val_metrics["accuracy"],
        hgnn_val_metrics["precision"],
        hgnn_val_metrics["recall"]
    ],
    marker=dict(color="red", opacity=0.7)
))

fig_metrics.update_layout(
    title="Validation Metrics Comparison",
    barmode="group",
    yaxis_title="Score",
    xaxis_title="Metric",
    height=500,
    template="plotly_white"
)

fig_metrics.show()

print("\n" + "="*100)
print("Comparison complete!")


Model Comparison: GNN vs HGNN

   Metric      GNN     HGNN
  ROC-AUC   0.9189   0.9404
 F1 Score   0.8437   0.8718
 Accuracy   0.8447   0.8694
Precision   0.8636   0.8698
   Recall   0.8247   0.8738
      MSE   0.1148   0.0970
 Val Loss 0.368279 0.316813

Difference (HGNN - GNN):
  roc_auc: +0.0214
  f1: +0.0281
  accuracy: +0.0247
  precision: +0.0062
  recall: +0.0492
  mse: -0.0178
  val_loss: -0.051466



Comparison complete!


In [60]:
# Save comprehensive comparison report
print("Saving comprehensive comparison report...")

comparison_report = {
    "timestamp": timestamp,
    "models": {
        "gnn": {
            "name": "Simple Graph Neural Network",
            "config": gnn_summary["model_config"],
            "optimizer": gnn_summary["optimizer_config"],
            "training": gnn_summary["training_config"],
            "metrics": gnn_val_metrics,
            "best_val_loss": gnn_best_val_loss,
            "checkpoint": str(gnn_exp_dir)
        },
        "hgnn": {
            "name": "Hypergraph Neural Network",
            "config": hgnn_summary["model_config"],
            "optimizer": hgnn_summary["optimizer_config"],
            "training": hgnn_summary["training_config"],
            "metrics": hgnn_val_metrics,
            "best_val_loss": hgnn_best_val_loss,
            "checkpoint": str(hgnn_exp_dir)
        }
    },
    "comparison": {
        "better_model": "HGNN" if hgnn_best_val_loss < gnn_best_val_loss else "GNN",
        "metric_differences": {
            "roc_auc": float(hgnn_val_metrics["roc_auc"] - gnn_val_metrics["roc_auc"]),
            "f1": float(hgnn_val_metrics["f1"] - gnn_val_metrics["f1"]),
            "accuracy": float(hgnn_val_metrics["accuracy"] - gnn_val_metrics["accuracy"]),
            "precision": float(hgnn_val_metrics["precision"] - gnn_val_metrics["precision"]),
            "recall": float(hgnn_val_metrics["recall"] - gnn_val_metrics["recall"]),
            "mse": float(hgnn_val_metrics["mse"] - gnn_val_metrics["mse"]),
            "val_loss": float(hgnn_best_val_loss - gnn_best_val_loss)
        },
        "training_epochs": {
            "gnn": len(gnn_history["train_loss"]),
            "hgnn": len(hgnn_history["train_loss"])
        }
    },
    "analysis": {
        "gnn_advantage": "Simpler model, faster training, lower memory footprint, efficient sparse operations",
        "hgnn_advantage": "Captures hypergraph structure, uses full outfit relationships, more expressive representations",
        "recommendation": "Use HGNN for production if resource-constrained environments allow; GNN for real-time applications"
    }
}

# Save report to both directories for easy access
report_path = gnn_exp_dir.parent / "comparison_report.json"
with open(report_path, "w") as f:
    json.dump(comparison_report, f, indent=2)
print(f"✓ Comparison report saved: {report_path}")

# Also save in HGNN directory
report_path_hgnn = hgnn_exp_dir.parent / "comparison_report.json"
with open(report_path_hgnn, "w") as f:
    json.dump(comparison_report, f, indent=2)

print("\n" + "="*100)
print("SUMMARY")
print("="*100)
print(f"\n✓ GNN Model: {gnn_exp_dir}")
print(f"  - Best Val Loss: {gnn_best_val_loss:.6f}")
print(f"  - ROC-AUC: {gnn_val_metrics['roc_auc']:.4f}")
print(f"  - F1 Score: {gnn_val_metrics['f1']:.4f}")

print(f"\n✓ HGNN Model: {hgnn_exp_dir}")
print(f"  - Best Val Loss: {hgnn_best_val_loss:.6f}")
print(f"  - ROC-AUC: {hgnn_val_metrics['roc_auc']:.4f}")
print(f"  - F1 Score: {hgnn_val_metrics['f1']:.4f}")

better = "HGNN" if hgnn_best_val_loss < gnn_best_val_loss else "GNN"
loss_improvement = abs(hgnn_best_val_loss - gnn_best_val_loss)
print(f"\n✓ Better Model: {better}")
print(f"  - Loss Difference: {loss_improvement:.6f}")

print(f"\n✓ Comparison Report: {report_path}")
print("="*100)

Saving comprehensive comparison report...
✓ Comparison report saved: ..\experiments_fashion_score\gnn_models\comparison_report.json

SUMMARY

✓ GNN Model: ..\experiments_fashion_score\gnn_models\simple_gnn_h256_g128_l2_h4_dr20_adamw_lr1e-4_wd1e-5_20260120_194208
  - Best Val Loss: 0.368279
  - ROC-AUC: 0.9189
  - F1 Score: 0.8437

✓ HGNN Model: ..\experiments_fashion_score\hgnn_models\simple_hgnn_h128_g64_l2_h2_dr20_adamw_lr1e-3_wd1e-5_20260120_194208
  - Best Val Loss: 0.316813
  - ROC-AUC: 0.9404
  - F1 Score: 0.8718

✓ Better Model: HGNN
  - Loss Difference: 0.051466

✓ Comparison Report: ..\experiments_fashion_score\gnn_models\comparison_report.json


In [61]:
print("\n" + "="*100)
print("📊 COMPREHENSIVE MODEL COMPARISON SUMMARY")
print("="*100 + "\n")

# 1. Overall Winner
print("🏆 OVERALL PERFORMANCE")
print("-" * 100)
better = "HGNN" if hgnn_best_val_loss < gnn_best_val_loss else "GNN"
loss_diff = abs(hgnn_best_val_loss - gnn_best_val_loss)
loss_improvement = (loss_diff / gnn_best_val_loss * 100) if better == "HGNN" else (loss_diff / hgnn_best_val_loss * 100)
print(f"✓ Winner: {better} Model")
print(f"  - Validation Loss Difference: {loss_diff:.6f} ({loss_improvement:.2f}% improvement)\n")

# 2. Detailed Metrics Comparison
print("📈 METRICS COMPARISON")
print("-" * 100)
metrics_list = ["roc_auc", "f1", "accuracy", "precision", "recall", "mse"]
comparison_data = {}

for metric in metrics_list:
    gnn_val = gnn_val_metrics[metric]
    hgnn_val = hgnn_val_metrics[metric]
    diff = hgnn_val - gnn_val
    
    # Determine which is better (higher is better for most, lower is better for MSE)
    if metric == "mse":
        winner = "GNN" if gnn_val < hgnn_val else "HGNN"
        symbol = "↓" if diff < 0 else "↑"
    else:
        winner = "HGNN" if hgnn_val > gnn_val else "GNN"
        symbol = "↑" if diff > 0 else "↓"
    
    comparison_data[metric] = {
        "gnn": gnn_val,
        "hgnn": hgnn_val,
        "diff": diff,
        "winner": winner
    }
    
    print(f"{metric.upper():12s} | GNN: {gnn_val:8.4f} | HGNN: {hgnn_val:8.4f} | Δ: {diff:+8.4f} {symbol} | Winner: {winner}")

print()

# 3. Training Efficiency
print("⚡ TRAINING EFFICIENCY")
print("-" * 100)
gnn_epochs = len(gnn_history["train_loss"])
hgnn_epochs = len(hgnn_history["train_loss"])
print(f"GNN Epochs:  {gnn_epochs} epochs to convergence")
print(f"HGNN Epochs: {hgnn_epochs} epochs to convergence")
print(f"Training Speed: {'GNN' if gnn_epochs <= hgnn_epochs else 'HGNN'} converged faster ({min(gnn_epochs, hgnn_epochs)} epochs)")
print()

# 4. Architecture Comparison
print("🏗️  ARCHITECTURE COMPARISON")
print("-" * 100)
gnn_params = sum(p.numel() for p in gnn_model.parameters())
hgnn_params = sum(p.numel() for p in hgnn_model.parameters())
param_ratio = hgnn_params / gnn_params
print(f"GNN Parameters:  {gnn_params:,}")
print(f"HGNN Parameters: {hgnn_params:,}")
print(f"Model Size Ratio: HGNN/GNN = {param_ratio:.2f}x")
print(f"Size Advantage: {'GNN' if gnn_params <= hgnn_params else 'HGNN'} (more parameters = more capacity)")
print()

# 5. Key Findings
print("🔍 KEY FINDINGS")
print("-" * 100)

# Find which metrics favor HGNN
hgnn_wins = sum(1 for m, d in comparison_data.items() if d["winner"] == "HGNN" and m != "mse")
hgnn_wins += 1 if comparison_data["mse"]["winner"] == "HGNN" else 0  # Lower MSE is better

gnn_wins = len(metrics_list) - hgnn_wins

print(f"✓ HGNN wins in {hgnn_wins}/{len(metrics_list)} metrics")
print(f"✓ GNN wins in {gnn_wins}/{len(metrics_list)} metrics")

# Specific analysis
if comparison_data["roc_auc"]["winner"] == "HGNN":
    print(f"✓ HGNN has better discriminative ability (ROC-AUC: {hgnn_val_metrics['roc_auc']:.4f} vs {gnn_val_metrics['roc_auc']:.4f})")
else:
    print(f"✓ GNN has better discriminative ability (ROC-AUC: {gnn_val_metrics['roc_auc']:.4f} vs {hgnn_val_metrics['roc_auc']:.4f})")

if comparison_data["f1"]["winner"] == "HGNN":
    print(f"✓ HGNN has better precision-recall balance (F1: {hgnn_val_metrics['f1']:.4f} vs {gnn_val_metrics['f1']:.4f})")
else:
    print(f"✓ GNN has better precision-recall balance (F1: {gnn_val_metrics['f1']:.4f} vs {hgnn_val_metrics['f1']:.4f})")

print()

# 6. Recommendations
print("💡 RECOMMENDATIONS")
print("-" * 100)
if better == "HGNN":
    print("✓ HGNN Model Recommended")
    print("  - Advantages:")
    print("    • Better captures outfit hypergraph relationships")
    print("    • Higher discriminative ability")
    print("    • Improved metrics across most dimensions")
    print("  - Trade-off:")
    print(f"    • {param_ratio:.2f}x more parameters ({hgnn_params:,} vs {gnn_params:,})")
    print(f"    • Slightly higher training time ({hgnn_epochs} vs {gnn_epochs} epochs)")
    print("  - Best For: Production systems with adequate computational resources")
else:
    print("✓ GNN Model Recommended")
    print("  - Advantages:")
    print("    • Simpler architecture, easier to understand and maintain")
    print(f"    • {1/param_ratio:.2f}x fewer parameters")
    print(f"    • Faster convergence ({gnn_epochs} vs {hgnn_epochs} epochs)")
    print("    • Lower memory footprint")
    print("  - Best For: Real-time applications and resource-constrained environments")

print()
print("📁 Saved Models & Reports:")
print(f"  GNN:  {gnn_exp_dir}")
print(f"  HGNN: {hgnn_exp_dir}")
print(f"  Report: {report_path}")
print("="*100)


📊 COMPREHENSIVE MODEL COMPARISON SUMMARY

🏆 OVERALL PERFORMANCE
----------------------------------------------------------------------------------------------------
✓ Winner: HGNN Model
  - Validation Loss Difference: 0.051466 (13.97% improvement)

📈 METRICS COMPARISON
----------------------------------------------------------------------------------------------------
ROC_AUC      | GNN:   0.9189 | HGNN:   0.9404 | Δ:  +0.0214 ↑ | Winner: HGNN
F1           | GNN:   0.8437 | HGNN:   0.8718 | Δ:  +0.0281 ↑ | Winner: HGNN
ACCURACY     | GNN:   0.8447 | HGNN:   0.8694 | Δ:  +0.0247 ↑ | Winner: HGNN
PRECISION    | GNN:   0.8636 | HGNN:   0.8698 | Δ:  +0.0062 ↑ | Winner: HGNN
RECALL       | GNN:   0.8247 | HGNN:   0.8738 | Δ:  +0.0492 ↑ | Winner: HGNN
MSE          | GNN:   0.1148 | HGNN:   0.0970 | Δ:  -0.0178 ↓ | Winner: HGNN

⚡ TRAINING EFFICIENCY
----------------------------------------------------------------------------------------------------
GNN Epochs:  27 epochs to convergence
HGNN